In [4]:
# ============================================================
# STEP 14 — KNOWLEDGE EDITING
# CELL 1 — DISCOVER INPUT FILES
# ============================================================

from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")

required_files = [
    "unified_products.parquet",
    "unified_reviews.parquet",
    "product_faiss_moi.index",
    "product_embedding_metadata.parquet",
    "step11_config.json",
    "bpr_model.pkl",
    "cf_mappings.pkl",
    "step12_config.json",
]

optional_files = [
    "step13_config.json",
    "step13_final_test.json",
]

print("=" * 80)
print("STEP 14 — INPUT DISCOVERY")
print("=" * 80)

found_files = {}

for filename in required_files + optional_files:

    matches = list(
        KAGGLE_INPUT.rglob(filename)
    )

    found_files[filename] = matches

    status = "FOUND" if matches else "MISSING"

    print(
        f"\n[{status}] {filename}"
    )

    for path in matches:
        print("   ", path)


print("\n" + "=" * 80)
print("REQUIRED FILE SUMMARY")
print("=" * 80)

missing_required = []

for filename in required_files:

    matches = found_files[filename]

    if matches:

        print(
            f"PASS | {filename}"
        )

    else:

        print(
            f"FAIL | {filename}"
        )

        missing_required.append(
            filename
        )


print("\n" + "=" * 80)

if missing_required:

    print(
        "MISSING REQUIRED FILES:",
        len(missing_required)
    )

    for filename in missing_required:
        print("-", filename)

else:

    print(
        "PASS: ALL REQUIRED "
        "STEP 14 INPUTS FOUND"
    )

STEP 14 — INPUT DISCOVERY

[FOUND] unified_products.parquet
    /kaggle/input/datasets/trnquangtriu/14dataset/unified_products.parquet

[FOUND] unified_reviews.parquet
    /kaggle/input/datasets/trnquangtriu/14dataset/unified_reviews.parquet

[FOUND] product_faiss_moi.index
    /kaggle/input/datasets/trnquangtriu/14dataset/product_faiss_moi.index

[FOUND] product_embedding_metadata.parquet
    /kaggle/input/datasets/trnquangtriu/14dataset/product_embedding_metadata.parquet

[FOUND] step11_config.json
    /kaggle/input/datasets/trnquangtriu/14dataset/step11_config.json

[FOUND] bpr_model.pkl
    /kaggle/input/datasets/trnquangtriu/14dataset/bpr_model.pkl

[FOUND] cf_mappings.pkl
    /kaggle/input/datasets/trnquangtriu/14dataset/cf_mappings.pkl

[FOUND] step12_config.json
    /kaggle/input/datasets/trnquangtriu/14dataset/step12_config.json

[FOUND] step13_config.json
    /kaggle/input/datasets/trnquangtriu/14dataset/step13_config.json

[FOUND] step13_final_test.json
    /kaggle/input/dat

In [5]:
# ============================================================
# STEP 14 — KNOWLEDGE EDITING
# CELL 2 — LOAD BASE DATA + CONFIGS
# ============================================================

import json
import pandas as pd
from pathlib import Path

DATA_DIR = Path(
    "/kaggle/input/datasets/trnquangtriu/14dataset"
)

PRODUCTS_PATH = (
    DATA_DIR / "unified_products.parquet"
)

REVIEWS_PATH = (
    DATA_DIR / "unified_reviews.parquet"
)

STEP11_CONFIG_PATH = (
    DATA_DIR / "step11_config.json"
)

STEP12_CONFIG_PATH = (
    DATA_DIR / "step12_config.json"
)

STEP13_CONFIG_PATH = (
    DATA_DIR / "step13_config.json"
)

STEP13_RESULT_PATH = (
    DATA_DIR / "step13_final_test.json"
)


def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


print("Loading products...")

products = pd.read_parquet(
    PRODUCTS_PATH
)

print("Loading reviews...")

reviews = pd.read_parquet(
    REVIEWS_PATH
)

step11_config = load_json(
    STEP11_CONFIG_PATH
)

step12_config = load_json(
    STEP12_CONFIG_PATH
)

step13_config = load_json(
    STEP13_CONFIG_PATH
)

step13_baseline = load_json(
    STEP13_RESULT_PATH
)


print("\n" + "=" * 80)
print("BASE DATA")
print("=" * 80)

print(
    "PRODUCTS SHAPE:",
    products.shape
)

print(
    "REVIEWS SHAPE:",
    reviews.shape
)

print(
    "\nPRODUCT ID UNIQUE:",
    products["product_id"].nunique()
)

print(
    "REVIEW ID UNIQUE:",
    reviews["review_id"].nunique()
)

print(
    "\nSTEP 11:",
    step11_config.get(
        "status",
        "status not stored"
    )
)

print(
    "STEP 12:",
    step12_config.get(
        "status",
        "status not stored"
    )
)

print(
    "STEP 13:",
    step13_config.get(
        "status",
        "status not stored"
    )
)

print(
    "\nSTEP 13 BASELINE MODE:",
    step13_baseline.get("mode")
)

print(
    "STEP 13 BASELINE PRODUCTS:",
    len(
        step13_baseline.get(
            "products",
            []
        )
    )
)

print("\nPASS: BASE DATA LOADED")

Loading products...
Loading reviews...

BASE DATA
PRODUCTS SHAPE: (164926, 60)
REVIEWS SHAPE: (767179, 16)

PRODUCT ID UNIQUE: 164926
REVIEW ID UNIQUE: 767179

STEP 11: status not stored
STEP 12: completed
STEP 13: end_to_end_test_completed

STEP 13 BASELINE MODE: hybrid_bpr
STEP 13 BASELINE PRODUCTS: 3

PASS: BASE DATA LOADED


In [6]:
%pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.0 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [7]:
# ============================================================
# CELL 3 — LOAD + VALIDATE FAISS
# ============================================================

import faiss
import pandas as pd

FAISS_PATH = (
    DATA_DIR / "product_faiss_moi.index"
)

METADATA_PATH = (
    DATA_DIR
    / "product_embedding_metadata.parquet"
)


print("Loading FAISS index...")

faiss_index = faiss.read_index(
    str(FAISS_PATH)
)

print("Loading embedding metadata...")

embedding_metadata = pd.read_parquet(
    METADATA_PATH
)


print("\n" + "=" * 80)
print("FAISS")
print("=" * 80)

print(
    "INDEX TYPE:",
    type(faiss_index).__name__
)

print(
    "VECTOR COUNT:",
    faiss_index.ntotal
)

print(
    "VECTOR DIMENSION:",
    faiss_index.d
)

print(
    "METADATA SHAPE:",
    embedding_metadata.shape
)


# ----------------------------
# Validation
# ----------------------------

assert (
    faiss_index.ntotal
    == len(embedding_metadata)
), (
    "FAISS vector count != "
    "metadata row count"
)

assert (
    faiss_index.ntotal
    == len(products)
), (
    "FAISS vector count != "
    "product count"
)

assert "faiss_row_id" in (
    embedding_metadata.columns
)

faiss_ids = (
    embedding_metadata["faiss_row_id"]
    .astype(int)
    .to_numpy()
)

expected_ids = range(
    len(embedding_metadata)
)

assert (
    faiss_ids.min() == 0
)

assert (
    faiss_ids.max()
    == len(embedding_metadata) - 1
)

assert (
    len(set(faiss_ids))
    == len(embedding_metadata)
)

assert all(
    actual == expected
    for actual, expected
    in zip(
        faiss_ids,
        expected_ids
    )
)


metadata_by_faiss_row = (
    embedding_metadata
    .set_index("faiss_row_id")
)


print("\nPASS: FAISS VALIDATED")

print(
    "ROWS:",
    faiss_index.ntotal
)

print(
    "DIM:",
    faiss_index.d
)

print(
    "FAISS ROW RANGE:",
    int(faiss_ids.min()),
    "->",
    int(faiss_ids.max())
)

Loading FAISS index...
Loading embedding metadata...

FAISS
INDEX TYPE: IndexFlatIP
VECTOR COUNT: 164926
VECTOR DIMENSION: 384
METADATA SHAPE: (164926, 17)

PASS: FAISS VALIDATED
ROWS: 164926
DIM: 384
FAISS ROW RANGE: 0 -> 164925


In [8]:
# ============================================================
# CELL 4 — INSPECT EMBEDDING CONFIG
# ============================================================

print("=" * 80)
print("STEP 11 CONFIG")
print("=" * 80)

print(
    json.dumps(
        step11_config,
        ensure_ascii=False,
        indent=2
    )
)


embedding_model_name = (
    step11_config.get(
        "embedding_model"
    )
    or step11_config.get(
        "model_name"
    )
    or (
        "sentence-transformers/"
        "paraphrase-multilingual-MiniLM-L12-v2"
    )
)


print("\nEMBEDDING MODEL:")
print(
    embedding_model_name
)

print(
    "\nFAISS DIMENSION:",
    faiss_index.d
)

STEP 11 CONFIG
{
  "step": "step11_product_embedding_faiss",
  "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "embedding_dimension": 384,
  "model_max_sequence_length": 128,
  "embedding_text_column": "embedding_text_v2",
  "normalization": true,
  "faiss_index_type": "IndexFlatIP",
  "similarity": "cosine",
  "total_products": 164926,
  "chunk_size": 4096,
  "batch_size": 64,
  "source_dataset": "step10_unified_products",
  "faiss_mapping_column": "faiss_row_id"
}

EMBEDDING MODEL:
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

FAISS DIMENSION: 384


In [9]:
%pip install -q implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 61.7 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [10]:
# ============================================================
# CELL 5 — LOAD + VALIDATE BPR
# ============================================================

import pickle
import numpy as np

BPR_PATH = (
    DATA_DIR / "bpr_model.pkl"
)

MAPPING_PATH = (
    DATA_DIR / "cf_mappings.pkl"
)


with open(
    BPR_PATH,
    "rb"
) as f:

    bpr_model = pickle.load(f)


with open(
    MAPPING_PATH,
    "rb"
) as f:

    cf_mappings = pickle.load(f)


user_to_idx = (
    cf_mappings["user_to_idx"]
)

item_to_idx = (
    cf_mappings["item_to_idx"]
)

idx_to_user = (
    cf_mappings["idx_to_user"]
)

idx_to_item = (
    cf_mappings["idx_to_item"]
)


print("=" * 80)
print("BPR")
print("=" * 80)

print(
    "MODEL:",
    type(bpr_model).__name__
)

print(
    "USER FACTORS:",
    bpr_model.user_factors.shape
)

print(
    "ITEM FACTORS:",
    bpr_model.item_factors.shape
)


# ----------------------------
# Validation
# ----------------------------

assert (
    len(user_to_idx)
    == bpr_model.user_factors.shape[0]
)

assert (
    len(item_to_idx)
    == bpr_model.item_factors.shape[0]
)

assert (
    len(idx_to_user)
    == bpr_model.user_factors.shape[0]
)

assert (
    len(idx_to_item)
    == bpr_model.item_factors.shape[0]
)

assert (
    bpr_model.user_factors.shape[1]
    == bpr_model.item_factors.shape[1]
)

assert np.isfinite(
    bpr_model.user_factors
).all()

assert np.isfinite(
    bpr_model.item_factors
).all()


product_ids = set(
    products["product_id"].astype(str)
)

bpr_product_ids = set(
    str(x)
    for x in idx_to_item
)

missing_bpr_products = (
    bpr_product_ids
    - product_ids
)


print(
    "\nBPR USERS:",
    len(user_to_idx)
)

print(
    "BPR ITEMS:",
    len(item_to_idx)
)

print(
    "LATENT DIMENSION:",
    bpr_model.user_factors.shape[1]
)

print(
    "BPR ITEMS MISSING FROM PRODUCTS:",
    len(missing_bpr_products)
)

assert (
    len(missing_bpr_products) == 0
)

print(
    "\nPASS: BPR + MAPPINGS VALIDATED"
)

/usr/local/lib/python3.12/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(


BPR
MODEL: BayesianPersonalizedRanking
USER FACTORS: (26903, 65)
ITEM FACTORS: (15546, 65)

BPR USERS: 26903
BPR ITEMS: 15546
LATENT DIMENSION: 65
BPR ITEMS MISSING FROM PRODUCTS: 0

PASS: BPR + MAPPINGS VALIDATED


In [11]:
# ============================================================
# STEP 14 — KNOWLEDGE EDITING
# CELL 6 — LOAD QUERY EMBEDDING MODEL
# ============================================================

from sentence_transformers import SentenceTransformer
import numpy as np

print("=" * 80)
print("LOADING EMBEDDING MODEL")
print("=" * 80)

print("MODEL:")
print(embedding_model_name)

product_query_model = SentenceTransformer(
    embedding_model_name
)

# Phải giống Step 11
product_query_model.max_seq_length = int(
    step11_config.get(
        "model_max_sequence_length",
        128
    )
)

embedding_dim = (
    product_query_model
    .get_sentence_embedding_dimension()
)

print(
    "\nMODEL DIMENSION:",
    embedding_dim
)

print(
    "FAISS DIMENSION:",
    faiss_index.d
)

print(
    "MAX SEQ LENGTH:",
    product_query_model.max_seq_length
)

assert embedding_dim == faiss_index.d, (
    f"Embedding dim {embedding_dim} "
    f"!= FAISS dim {faiss_index.d}"
)

print(
    "\nPASS: QUERY EMBEDDING MODEL READY"
)

LOADING EMBEDDING MODEL
MODEL:
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


MODEL DIMENSION: 384
FAISS DIMENSION: 384
MAX SEQ LENGTH: 128

PASS: QUERY EMBEDDING MODEL READY


/tmp/ipykernel_58/1735692235.py:30: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  .get_sentence_embedding_dimension()


In [12]:
# ============================================================
# CELL 7 — FAISS RETRIEVAL SMOKE TEST
# ============================================================

TEST_QUERY = (
    "microSD 64GB memory card"
)

print("=" * 80)
print("FAISS RETRIEVAL SMOKE TEST")
print("=" * 80)

print(
    "QUERY:",
    TEST_QUERY
)

query_vector = (
    product_query_model.encode(
        [TEST_QUERY],
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    .astype("float32")
)

print(
    "\nQUERY VECTOR SHAPE:",
    query_vector.shape
)

scores, indices = (
    faiss_index.search(
        query_vector,
        5
    )
)

print("\nTOP 5 FAISS RESULTS")
print("-" * 80)

retrieval_test_results = []

for rank, (score, idx) in enumerate(
    zip(
        scores[0],
        indices[0]
    ),
    start=1
):

    idx = int(idx)

    assert idx >= 0

    meta = (
        metadata_by_faiss_row
        .loc[idx]
    )

    product_id = str(
        meta["product_id"]
    )

    product_match = products[
        products["product_id"].astype(str)
        == product_id
    ]

    assert len(product_match) == 1, (
        f"Product mapping error: "
        f"{product_id}"
    )

    product_row = (
        product_match.iloc[0]
    )

    result = {
        "rank": rank,
        "faiss_row_id": idx,
        "product_id": product_id,
        "title": product_row.get(
            "title"
        ),
        "score": float(score)
    }

    retrieval_test_results.append(
        result
    )

    print(
        f"\n#{rank}"
    )

    print(
        "FAISS ROW:",
        idx
    )

    print(
        "PRODUCT ID:",
        product_id
    )

    print(
        "TITLE:",
        product_row.get("title")
    )

    print(
        "SIMILARITY:",
        round(
            float(score),
            4
        )
    )


print(
    "\nPASS: EMBEDDING -> FAISS -> "
    "METADATA -> PRODUCT"
)

FAISS RETRIEVAL SMOKE TEST
QUERY: microSD 64GB memory card

QUERY VECTOR SHAPE: (1, 384)

TOP 5 FAISS RESULTS
--------------------------------------------------------------------------------

#1
FAISS ROW: 164363
PRODUCT ID: shopee_19591042905
TITLE: None
SIMILARITY: 0.8349

#2
FAISS ROW: 1254
PRODUCT ID: amazon_B09HKJY2QB
TITLE: None
SIMILARITY: 0.8274

#3
FAISS ROW: 6869
PRODUCT ID: amazon_B0143RTCD6
TITLE: None
SIMILARITY: 0.8086

#4
FAISS ROW: 2185
PRODUCT ID: amazon_B07XDCZ9J3
TITLE: None
SIMILARITY: 0.8055

#5
FAISS ROW: 51
PRODUCT ID: amazon_B0BRZPCQNJ
TITLE: None
SIMILARITY: 0.8018

PASS: EMBEDDING -> FAISS -> METADATA -> PRODUCT


In [13]:
# ============================================================
# CELL 8 — INITIALIZE KNOWLEDGE EDIT STORE
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone

STEP14_OUTPUT_DIR = Path(
    "/kaggle/working/step14"
)

STEP14_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

KNOWLEDGE_EDIT_PATH = (
    STEP14_OUTPUT_DIR
    / "knowledge_edits.json"
)


# Các trường được phép edit
EDITABLE_FIELDS = {
    "title",
    "description",
    "product_features",
    "price",
    "currency",
    "product_url"
}


# Các trường tuyệt đối không edit
PROTECTED_FIELDS = {
    "product_id",
    "source",
    "faiss_row_id",
    "review_id",
    "user_id"
}


def load_knowledge_edits():

    if not KNOWLEDGE_EDIT_PATH.exists():
        return []

    with open(
        KNOWLEDGE_EDIT_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(
            "Knowledge edit store "
            "must contain a list."
        )

    return data


def save_knowledge_edits(edits):

    with open(
        KNOWLEDGE_EDIT_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            edits,
            f,
            ensure_ascii=False,
            indent=2,
            default=str
        )


if not KNOWLEDGE_EDIT_PATH.exists():

    save_knowledge_edits([])


knowledge_edits = (
    load_knowledge_edits()
)


print("=" * 80)
print("KNOWLEDGE EDIT STORE")
print("=" * 80)

print(
    "PATH:",
    KNOWLEDGE_EDIT_PATH
)

print(
    "EXISTS:",
    KNOWLEDGE_EDIT_PATH.exists()
)

print(
    "CURRENT EDIT COUNT:",
    len(knowledge_edits)
)

print(
    "\nEDITABLE FIELDS:"
)

for field in sorted(
    EDITABLE_FIELDS
):
    print("-", field)


print(
    "\nPROTECTED FIELDS:"
)

for field in sorted(
    PROTECTED_FIELDS
):
    print("-", field)


print(
    "\nPASS: KNOWLEDGE EDIT STORE READY"
)

KNOWLEDGE EDIT STORE
PATH: /kaggle/working/step14/knowledge_edits.json
EXISTS: True
CURRENT EDIT COUNT: 0

EDITABLE FIELDS:
- currency
- description
- price
- product_features
- product_url
- title

PROTECTED FIELDS:
- faiss_row_id
- product_id
- review_id
- source
- user_id

PASS: KNOWLEDGE EDIT STORE READY


In [14]:
# ============================================================
# CELL 9 — INSPECT PRODUCT KNOWLEDGE SCHEMA
# ============================================================

print("=" * 80)
print("PRODUCT KNOWLEDGE SCHEMA")
print("=" * 80)

print("\nALL COLUMNS:")

for i, col in enumerate(
    products.columns,
    start=1
):
    print(
        f"{i:02d}. {col}"
    )


TEST_PRODUCT_ID = (
    "amazon_B09HKJY2QB"
)

test_rows = products[
    products["product_id"].astype(str)
    == TEST_PRODUCT_ID
]

assert len(test_rows) == 1

test_product = (
    test_rows.iloc[0]
)


print("\n" + "=" * 80)
print("TEST PRODUCT")
print("=" * 80)

print(
    "PRODUCT ID:",
    TEST_PRODUCT_ID
)


# Các field có khả năng chứa knowledge
candidate_fields = [
    "title",
    "product_name",
    "name",
    "smart_title",
    "description",
    "product_features",
    "features",
    "price",
    "smart_price",
    "currency",
    "smart_currency",
    "average_rating",
    "smart_rating",
    "review_count",
    "smart_review_count",
    "product_url"
]


for field in candidate_fields:

    if field not in products.columns:
        continue

    value = test_product[field]

    print(
        f"\n[{field}]"
    )

    print(
        str(value)[:1000]
    )


print(
    "\nPASS: PRODUCT KNOWLEDGE INSPECTED"
)

PRODUCT KNOWLEDGE SCHEMA

ALL COLUMNS:
01. product_id
02. product_name
03. category
04. subcategory
05. brand
06. price
07. currency
08. average_rating
09. review_count
10. sold_count
11. description
12. image_url
13. product_url
14. source
15. language
16. smart_product_name
17. smart_source
18. smart_currency
19. smart_price
20. smart_rating
21. smart_review_count
22. product_family
23. product_type
24. classification_method
25. matched_rule
26. has_valid_price
27. has_valid_currency
28. price_filter_eligible
29. quality_issue_count
30. quality_flags_json
31. product_family_v1
32. product_type_v1
33. classification_method_v1
34. matched_rule_v1
35. product_family_v2
36. product_type_v2
37. classification_method_v2
38. classification_reason_v2
39. etl_review_rows
40. etl_review_text_rows
41. etl_review_evidence_rows
42. has_review_evidence
43. product_url_available
44. product_type_candidate
45. product_family_candidate
46. classification_requires_validation
47. top5_evidence_ready
48

In [15]:
# ============================================================
# CELL 10 — INSPECT STEP 13 BASELINE PRODUCT
# ============================================================

print("=" * 80)
print("STEP 13 BASELINE")
print("=" * 80)

baseline_products = (
    step13_baseline.get(
        "products",
        []
    )
)

print(
    "BASELINE PRODUCT COUNT:",
    len(baseline_products)
)


target_baseline = None

for product in baseline_products:

    if (
        str(product.get("product_id"))
        == TEST_PRODUCT_ID
    ):

        target_baseline = product
        break


if target_baseline is None:

    print(
        "\nTarget product not present "
        "in Step 13 baseline."
    )

else:

    print(
        "\nTARGET PRODUCT FOUND:"
    )

    print(
        json.dumps(
            target_baseline,
            ensure_ascii=False,
            indent=2,
            default=str
        )
    )

STEP 13 BASELINE
BASELINE PRODUCT COUNT: 3

TARGET PRODUCT FOUND:
{
  "product_id": "amazon_B09HKJY2QB",
  "title": "Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)",
  "source": "amazon",
  "price": 6.49,
  "currency": "USD",
  "product_average_rating": 4.6,
  "metadata_rating_count": 15428,
  "linked_review_count": 33,
  "retrieved_evidence_count": 3,
  "product_url": null,
  "description": "",
  "features": "Micro SD Card 64GB. Fast Read/Write Speed up to: 95MB/s and 30MB/s respectively for high resolution photo capturing. Extended Capacity for pictures, music, documents\n4K UHD Capable and Full HD Ready with UHS speed class 3 (U3) and video speed class 30 (V30). Cool travel gadgets for action cameras, DSLR, drones, laptops and more\nRated Application Class 1 (A1) for faster app loading and enhanced app performance. Great storage accessories for...",
  "reviews": [
    {
      "revie

# Chuẩn hóa schema Knowledge Editing

In [16]:
# ============================================================
# CELL 11 — DEFINE KNOWLEDGE EDIT SCHEMA
# ============================================================

from datetime import datetime, timezone
import uuid
import math
import pandas as pd
import numpy as np


EDITABLE_FIELDS = {
    "product_name",
    "description",
    "product_features",
    "price",
    "currency",
    "product_url"
}

PROTECTED_FIELDS = {
    "product_id",
    "source",
    "average_rating",
    "review_count",
    "smart_rating",
    "smart_review_count",
    "faiss_row_id",
    "review_id",
    "user_id"
}


def normalize_value(value):
    """
    Chuyển NaN/None về None để lưu JSON an toàn.
    """

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(value, np.generic):
        return value.item()

    return value


print("=" * 80)
print("KNOWLEDGE EDIT SCHEMA")
print("=" * 80)

print("\nEDITABLE:")
for field in sorted(EDITABLE_FIELDS):
    print("-", field)

print("\nPROTECTED:")
for field in sorted(PROTECTED_FIELDS):
    print("-", field)

print(
    "\nPASS: KNOWLEDGE EDIT SCHEMA DEFINED"
)

KNOWLEDGE EDIT SCHEMA

EDITABLE:
- currency
- description
- price
- product_features
- product_name
- product_url

PROTECTED:
- average_rating
- faiss_row_id
- product_id
- review_count
- review_id
- smart_rating
- smart_review_count
- source
- user_id

PASS: KNOWLEDGE EDIT SCHEMA DEFINED


# Hàm tạo Knowledge Edit

In [17]:
# ============================================================
# CELL 12 — CREATE KNOWLEDGE EDIT FUNCTION
# ============================================================

def create_knowledge_edit(
    product_id,
    field,
    new_value,
    reason,
    evidence=None,
    provenance="controlled_test",
    activate=True
):

    product_id = str(product_id)

    # -------------------------
    # Validate product
    # -------------------------

    matches = products[
        products["product_id"].astype(str)
        == product_id
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly 1 product "
            f"for {product_id}, found {len(matches)}"
        )

    # -------------------------
    # Validate field
    # -------------------------

    if field in PROTECTED_FIELDS:
        raise ValueError(
            f"Protected field cannot be edited: {field}"
        )

    if field not in EDITABLE_FIELDS:
        raise ValueError(
            f"Field not allowed: {field}"
        )

    if field not in products.columns:
        raise ValueError(
            f"Field does not exist in products: {field}"
        )

    # -------------------------
    # Get original value
    # -------------------------

    old_value = normalize_value(
        matches.iloc[0][field]
    )

    new_value = normalize_value(
        new_value
    )

    if old_value == new_value:
        raise ValueError(
            "New value is identical to old value."
        )

    # -------------------------
    # Version
    # -------------------------

    existing = [
        edit
        for edit in load_knowledge_edits()
        if (
            str(edit.get("product_id"))
            == product_id
            and edit.get("field") == field
        )
    ]

    version = len(existing) + 1

    # -------------------------
    # Create record
    # -------------------------

    edit = {
        "edit_id": str(uuid.uuid4()),
        "product_id": product_id,
        "field": field,

        "old_value": old_value,
        "new_value": new_value,

        "reason": str(reason),
        "evidence": evidence,
        "provenance": provenance,

        "created_at": datetime.now(
            timezone.utc
        ).isoformat(),

        "status": (
            "active"
            if activate
            else "inactive"
        ),

        "version": version
    }

    return edit


print(
    "PASS: create_knowledge_edit() READY"
)

PASS: create_knowledge_edit() READY


# Tạo Knowledge Edit thử nghiệm đầu tiên

In [18]:
# ============================================================
# CELL 13 — CREATE FIRST CONTROLLED KNOWLEDGE EDIT
# ============================================================

TEST_PRODUCT_ID = (
    "amazon_B09HKJY2QB"
)

TEST_DESCRIPTION = (
    "[CONTROLLED KE TEST] "
    "This description was added by the Step 14 "
    "Knowledge Editing evaluation layer."
)


first_edit = create_knowledge_edit(
    product_id=TEST_PRODUCT_ID,

    field="description",

    new_value=TEST_DESCRIPTION,

    reason=(
        "Controlled Step 14 experiment to test "
        "knowledge-edit insertion, locality, "
        "persistence and rollback."
    ),

    evidence={
        "type": "controlled_experiment",
        "verified_product_claim": False
    },

    provenance="controlled_test",

    activate=True
)


print("=" * 80)
print("PROPOSED KNOWLEDGE EDIT")
print("=" * 80)

print(
    json.dumps(
        first_edit,
        ensure_ascii=False,
        indent=2,
        default=str
    )
)

PROPOSED KNOWLEDGE EDIT
{
  "edit_id": "00f330af-10b3-4f70-9eeb-a0c815e60402",
  "product_id": "amazon_B09HKJY2QB",
  "field": "description",
  "old_value": null,
  "new_value": "[CONTROLLED KE TEST] This description was added by the Step 14 Knowledge Editing evaluation layer.",
  "reason": "Controlled Step 14 experiment to test knowledge-edit insertion, locality, persistence and rollback.",
  "evidence": {
    "type": "controlled_experiment",
    "verified_product_claim": false
  },
  "provenance": "controlled_test",
  "created_at": "2026-09-29T07:25:54.389825+00:00",
  "status": "active",
  "version": 1
}


# Commit edit vào Knowledge Edit Store

In [19]:
# ============================================================
# CELL 14 — COMMIT KNOWLEDGE EDIT
# ============================================================

knowledge_edits = (
    load_knowledge_edits()
)


# Tránh commit cùng edit_id hai lần
existing_ids = {
    edit["edit_id"]
    for edit in knowledge_edits
}


if first_edit["edit_id"] not in existing_ids:

    knowledge_edits.append(
        first_edit
    )

    save_knowledge_edits(
        knowledge_edits
    )


knowledge_edits = (
    load_knowledge_edits()
)


print("=" * 80)
print("KNOWLEDGE EDIT COMMITTED")
print("=" * 80)

print(
    "TOTAL EDITS:",
    len(knowledge_edits)
)

print(
    "LAST EDIT ID:",
    knowledge_edits[-1]["edit_id"]
)

print(
    "PRODUCT:",
    knowledge_edits[-1]["product_id"]
)

print(
    "FIELD:",
    knowledge_edits[-1]["field"]
)

print(
    "VERSION:",
    knowledge_edits[-1]["version"]
)

print(
    "STATUS:",
    knowledge_edits[-1]["status"]
)

print(
    "\nSTORE:",
    KNOWLEDGE_EDIT_PATH
)

print(
    "\nPASS: EDIT SAVED"
)

KNOWLEDGE EDIT COMMITTED
TOTAL EDITS: 1
LAST EDIT ID: 00f330af-10b3-4f70-9eeb-a0c815e60402
PRODUCT: amazon_B09HKJY2QB
FIELD: description
VERSION: 1
STATUS: active

STORE: /kaggle/working/step14/knowledge_edits.json

PASS: EDIT SAVED


# Hàm lấy Effective Knowledge

In [20]:
# ============================================================
# CELL 15 — APPLY KNOWLEDGE EDIT OVERLAY
# ============================================================

import copy


def get_original_product(product_id):

    product_id = str(product_id)

    matches = products[
        products["product_id"].astype(str)
        == product_id
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly 1 product for "
            f"{product_id}, found {len(matches)}"
        )

    product = (
        matches.iloc[0]
        .to_dict()
    )

    # Chuẩn hóa NaN
    for key, value in product.items():
        product[key] = normalize_value(value)

    return product


def get_active_edits(product_id):

    product_id = str(product_id)

    edits = [
        edit
        for edit in load_knowledge_edits()
        if (
            str(edit.get("product_id"))
            == product_id
            and edit.get("status") == "active"
        )
    ]

    # version thấp -> cao
    edits = sorted(
        edits,
        key=lambda x: (
            x.get("field", ""),
            int(x.get("version", 0))
        )
    )

    return edits


def get_effective_product_knowledge(
    product_id
):

    original = get_original_product(
        product_id
    )

    effective = copy.deepcopy(
        original
    )

    active_edits = get_active_edits(
        product_id
    )

    # Nếu cùng field có nhiều version,
    # version sau cùng sẽ override version trước.
    applied = {}

    for edit in active_edits:

        field = edit["field"]

        if field not in EDITABLE_FIELDS:
            continue

        effective[field] = (
            edit["new_value"]
        )

        applied[field] = {
            "edit_id": edit["edit_id"],
            "version": edit["version"],
            "provenance": edit["provenance"]
        }

    return {
        "product_id": str(product_id),
        "original": original,
        "effective": effective,
        "applied_edits": applied,
        "active_edit_count": len(
            active_edits
        )
    }


print(
    "PASS: KNOWLEDGE OVERLAY ENGINE READY"
)

PASS: KNOWLEDGE OVERLAY ENGINE READY


# Reliability Test

In [21]:
# ============================================================
# CELL 16 — RELIABILITY TEST
# ============================================================

edited_result = (
    get_effective_product_knowledge(
        TEST_PRODUCT_ID
    )
)

original_description = (
    edited_result["original"]
    .get("description")
)

effective_description = (
    edited_result["effective"]
    .get("description")
)


print("=" * 80)
print("KNOWLEDGE EDIT — RELIABILITY TEST")
print("=" * 80)

print(
    "PRODUCT:",
    TEST_PRODUCT_ID
)

print(
    "\nORIGINAL DESCRIPTION:"
)
print(
    original_description
)

print(
    "\nEFFECTIVE DESCRIPTION:"
)
print(
    effective_description
)

print(
    "\nAPPLIED EDITS:"
)

print(
    json.dumps(
        edited_result[
            "applied_edits"
        ],
        ensure_ascii=False,
        indent=2
    )
)


reliability_pass = (
    original_description is None
    and
    effective_description
    == TEST_DESCRIPTION
)


print(
    "\nRELIABILITY PASS:",
    reliability_pass
)

assert reliability_pass

print(
    "\nPASS: TARGET KNOWLEDGE "
    "WAS SUCCESSFULLY EDITED"
)

KNOWLEDGE EDIT — RELIABILITY TEST
PRODUCT: amazon_B09HKJY2QB

ORIGINAL DESCRIPTION:
None

EFFECTIVE DESCRIPTION:
[CONTROLLED KE TEST] This description was added by the Step 14 Knowledge Editing evaluation layer.

APPLIED EDITS:
{
  "description": {
    "edit_id": "00f330af-10b3-4f70-9eeb-a0c815e60402",
    "version": 1,
    "provenance": "controlled_test"
  }
}

RELIABILITY PASS: True

PASS: TARGET KNOWLEDGE WAS SUCCESSFULLY EDITED


# LOCALITY TEST


In [22]:
# ============================================================
# CELL 17 — LOCALITY TEST
# ============================================================

CONTROL_PRODUCT_ID = (
    "amazon_B0BRZPCQNJ"
)


control_original = (
    get_original_product(
        CONTROL_PRODUCT_ID
    )
)

control_result = (
    get_effective_product_knowledge(
        CONTROL_PRODUCT_ID
    )
)

control_effective = (
    control_result["effective"]
)


print("=" * 80)
print("KNOWLEDGE EDIT — LOCALITY TEST")
print("=" * 80)

print(
    "EDITED PRODUCT:",
    TEST_PRODUCT_ID
)

print(
    "CONTROL PRODUCT:",
    CONTROL_PRODUCT_ID
)


# Chỉ so sánh các field knowledge
fields_to_compare = [
    "product_name",
    "description",
    "product_features",
    "price",
    "currency",
    "product_url"
]


locality_results = {}


for field in fields_to_compare:

    before = normalize_value(
        control_original.get(field)
    )

    after = normalize_value(
        control_effective.get(field)
    )

    unchanged = (
        before == after
    )

    locality_results[field] = (
        unchanged
    )

    print(
        f"\n{field}:",
        "UNCHANGED"
        if unchanged
        else "CHANGED"
    )


locality_pass = all(
    locality_results.values()
)


print(
    "\nCONTROL ACTIVE EDITS:",
    control_result[
        "active_edit_count"
    ]
)

print(
    "LOCALITY PASS:",
    locality_pass
)


assert locality_pass


print(
    "\nPASS: UNRELATED PRODUCT "
    "REMAINED UNCHANGED"
)

KNOWLEDGE EDIT — LOCALITY TEST
EDITED PRODUCT: amazon_B09HKJY2QB
CONTROL PRODUCT: amazon_B0BRZPCQNJ

product_name: UNCHANGED

description: UNCHANGED

product_features: UNCHANGED

price: UNCHANGED

currency: UNCHANGED

product_url: UNCHANGED

CONTROL ACTIVE EDITS: 0
LOCALITY PASS: True

PASS: UNRELATED PRODUCT REMAINED UNCHANGED


# PERSISTENCE TEST


In [23]:
# ============================================================
# CELL 18 — PERSISTENCE TEST
# ============================================================

print("=" * 80)
print("KNOWLEDGE EDIT — PERSISTENCE TEST")
print("=" * 80)

# Đọc lại trực tiếp từ file
reloaded_edits = (
    load_knowledge_edits()
)

matching_edits = [
    edit
    for edit in reloaded_edits
    if (
        edit.get("edit_id")
        == first_edit["edit_id"]
    )
]


persistence_pass = (
    len(matching_edits) == 1
)


print(
    "EDIT STORE:",
    KNOWLEDGE_EDIT_PATH
)

print(
    "TOTAL EDITS AFTER RELOAD:",
    len(reloaded_edits)
)

print(
    "TARGET EDIT FOUND:",
    len(matching_edits) == 1
)


if matching_edits:

    persisted_edit = (
        matching_edits[0]
    )

    print(
        "EDIT ID:",
        persisted_edit["edit_id"]
    )

    print(
        "PRODUCT:",
        persisted_edit["product_id"]
    )

    print(
        "FIELD:",
        persisted_edit["field"]
    )

    print(
        "VERSION:",
        persisted_edit["version"]
    )

    print(
        "STATUS:",
        persisted_edit["status"]
    )


print(
    "\nPERSISTENCE PASS:",
    persistence_pass
)

assert persistence_pass

print(
    "\nPASS: KNOWLEDGE EDIT "
    "PERSISTED TO STORE"
)

KNOWLEDGE EDIT — PERSISTENCE TEST
EDIT STORE: /kaggle/working/step14/knowledge_edits.json
TOTAL EDITS AFTER RELOAD: 1
TARGET EDIT FOUND: True
EDIT ID: 00f330af-10b3-4f70-9eeb-a0c815e60402
PRODUCT: amazon_B09HKJY2QB
FIELD: description
VERSION: 1
STATUS: active

PERSISTENCE PASS: True

PASS: KNOWLEDGE EDIT PERSISTED TO STORE


# Tạo chức năng Activate / Deactivate

In [24]:
# ============================================================
# CELL 19 — EDIT STATUS / ROLLBACK FUNCTIONS
# ============================================================

def set_edit_status(
    edit_id,
    status
):

    if status not in {
        "active",
        "inactive"
    }:
        raise ValueError(
            "status must be "
            "'active' or 'inactive'"
        )

    edits = (
        load_knowledge_edits()
    )

    found = False

    for edit in edits:

        if (
            edit.get("edit_id")
            == edit_id
        ):

            edit["status"] = status

            edit["status_updated_at"] = (
                datetime.now(
                    timezone.utc
                ).isoformat()
            )

            found = True
            break

    if not found:
        raise ValueError(
            f"Edit ID not found: "
            f"{edit_id}"
        )

    save_knowledge_edits(
        edits
    )

    return found


def deactivate_edit(edit_id):

    return set_edit_status(
        edit_id,
        "inactive"
    )


def activate_edit(edit_id):

    return set_edit_status(
        edit_id,
        "active"
    )


print(
    "PASS: EDIT STATUS FUNCTIONS READY"
)

PASS: EDIT STATUS FUNCTIONS READY


# ROLLBACK TEST


In [25]:
# ============================================================
# CELL 20 — ROLLBACK TEST
# ============================================================

print("=" * 80)
print("KNOWLEDGE EDIT — ROLLBACK TEST")
print("=" * 80)

EDIT_ID = (
    first_edit["edit_id"]
)


# ---------------------------
# Before rollback
# ---------------------------

before_rollback = (
    get_effective_product_knowledge(
        TEST_PRODUCT_ID
    )
)

before_value = (
    before_rollback["effective"]
    .get("description")
)


print(
    "BEFORE ROLLBACK:"
)
print(
    before_value
)


# ---------------------------
# Deactivate
# ---------------------------

deactivate_edit(
    EDIT_ID
)


# ---------------------------
# After rollback
# ---------------------------

after_rollback = (
    get_effective_product_knowledge(
        TEST_PRODUCT_ID
    )
)

after_value = (
    after_rollback["effective"]
    .get("description")
)


print(
    "\nAFTER ROLLBACK:"
)
print(
    after_value
)

print(
    "\nACTIVE EDIT COUNT:",
    after_rollback[
        "active_edit_count"
    ]
)


rollback_pass = (
    after_value
    ==
    after_rollback[
        "original"
    ].get("description")
)


print(
    "\nROLLBACK PASS:",
    rollback_pass
)

assert rollback_pass


print(
    "\nPASS: ORIGINAL KNOWLEDGE "
    "RESTORED"
)

KNOWLEDGE EDIT — ROLLBACK TEST
BEFORE ROLLBACK:
[CONTROLLED KE TEST] This description was added by the Step 14 Knowledge Editing evaluation layer.

AFTER ROLLBACK:
None

ACTIVE EDIT COUNT: 0

ROLLBACK PASS: True

PASS: ORIGINAL KNOWLEDGE RESTORED


# Reactivate và kiểm tra lại

In [26]:
# ============================================================
# CELL 21 — REACTIVATE EDIT
# ============================================================

activate_edit(
    EDIT_ID
)


reactivated = (
    get_effective_product_knowledge(
        TEST_PRODUCT_ID
    )
)


reactivated_value = (
    reactivated["effective"]
    .get("description")
)


reactivation_pass = (
    reactivated_value
    == TEST_DESCRIPTION
)


print("=" * 80)
print("KNOWLEDGE EDIT — REACTIVATION")
print("=" * 80)

print(
    "EDIT ID:",
    EDIT_ID
)

print(
    "STATUS: active"
)

print(
    "\nEFFECTIVE DESCRIPTION:"
)
print(
    reactivated_value
)

print(
    "\nREACTIVATION PASS:",
    reactivation_pass
)

assert reactivation_pass

print(
    "\nPASS: EDIT REACTIVATED"
)

KNOWLEDGE EDIT — REACTIVATION
EDIT ID: 00f330af-10b3-4f70-9eeb-a0c815e60402
STATUS: active

EFFECTIVE DESCRIPTION:
[CONTROLLED KE TEST] This description was added by the Step 14 Knowledge Editing evaluation layer.

REACTIVATION PASS: True

PASS: EDIT REACTIVATED


# Tạo KE-aware Product Context

In [27]:
# ============================================================
# CELL 22 — BUILD KE-AWARE PRODUCT CONTEXT
# ============================================================

def build_ke_product_context(product_id):

    result = get_effective_product_knowledge(
        product_id
    )

    p = result["effective"]

    context = {
        "product_id": str(p.get("product_id")),

        "title": normalize_value(
            p.get("product_name")
        ),

        "source": normalize_value(
            p.get("source")
        ),

        "price": normalize_value(
            p.get("price")
        ),

        "currency": normalize_value(
            p.get("currency")
        ),

        "product_average_rating": normalize_value(
            p.get("average_rating")
        ),

        "metadata_rating_count": normalize_value(
            p.get("review_count")
        ),

        "product_url": normalize_value(
            p.get("product_url")
        ),

        "description": normalize_value(
            p.get("description")
        ),

        "features": normalize_value(
            p.get("product_features")
        ),

        "knowledge_edit": {
            "applied": (
                len(result["applied_edits"]) > 0
            ),

            "active_edit_count": (
                result["active_edit_count"]
            ),

            "fields": (
                result["applied_edits"]
            )
        }
    }

    return context


test_ke_context = (
    build_ke_product_context(
        TEST_PRODUCT_ID
    )
)


print("=" * 80)
print("KE-AWARE PRODUCT CONTEXT")
print("=" * 80)

print(
    json.dumps(
        test_ke_context,
        ensure_ascii=False,
        indent=2,
        default=str
    )
)

KE-AWARE PRODUCT CONTEXT
{
  "product_id": "amazon_B09HKJY2QB",
  "title": "Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)",
  "source": "amazon",
  "price": 6.49,
  "currency": "USD",
  "product_average_rating": 4.6,
  "metadata_rating_count": 15428,
  "product_url": null,
  "description": "[CONTROLLED KE TEST] This description was added by the Step 14 Knowledge Editing evaluation layer.",
  "features": "Micro SD Card 64GB. Fast Read/Write Speed up to: 95MB/s and 30MB/s respectively for high resolution photo capturing. Extended Capacity for pictures, music, documents\n4K UHD Capable and Full HD Ready with UHS speed class 3 (U3) and video speed class 30 (V30). Cool travel gadgets for action cameras, DSLR, drones, laptops and more\nRated Application Class 1 (A1) for faster app loading and enhanced app performance. Great storage accessories for tablets, smartphones, games consoles, andro

# So sánh RAG Context khi KE OFF và ON

In [28]:
# ============================================================
# CELL 23 — BEFORE / AFTER KE CONTEXT TEST
# ============================================================

print("=" * 80)
print("RAG CONTEXT — KE OFF vs KE ON")
print("=" * 80)


# ----------------------------------
# KE OFF
# ----------------------------------

deactivate_edit(
    EDIT_ID
)

context_ke_off = (
    build_ke_product_context(
        TEST_PRODUCT_ID
    )
)


# ----------------------------------
# KE ON
# ----------------------------------

activate_edit(
    EDIT_ID
)

context_ke_on = (
    build_ke_product_context(
        TEST_PRODUCT_ID
    )
)


print("\n--- KE OFF ---")

print(
    "DESCRIPTION:",
    context_ke_off["description"]
)

print(
    "EDIT APPLIED:",
    context_ke_off[
        "knowledge_edit"
    ]["applied"]
)


print("\n--- KE ON ---")

print(
    "DESCRIPTION:",
    context_ke_on["description"]
)

print(
    "EDIT APPLIED:",
    context_ke_on[
        "knowledge_edit"
    ]["applied"]
)


context_update_pass = (
    context_ke_off["description"] is None
    and
    context_ke_on["description"]
    == TEST_DESCRIPTION
)


print(
    "\nCONTEXT UPDATE PASS:",
    context_update_pass
)

assert context_update_pass

print(
    "\nPASS: RAG PRODUCT CONTEXT "
    "USES KNOWLEDGE EDIT"
)

RAG CONTEXT — KE OFF vs KE ON

--- KE OFF ---
DESCRIPTION: None
EDIT APPLIED: False

--- KE ON ---
DESCRIPTION: [CONTROLLED KE TEST] This description was added by the Step 14 Knowledge Editing evaluation layer.
EDIT APPLIED: True

CONTEXT UPDATE PASS: True

PASS: RAG PRODUCT CONTEXT USES KNOWLEDGE EDIT


# Kiểm tra chỉ field được edit thay đổi

In [29]:
# ============================================================
# CELL 24 — FIELD ISOLATION TEST
# ============================================================

print("=" * 80)
print("KNOWLEDGE EDIT — FIELD ISOLATION TEST")
print("=" * 80)


stable_fields = [
    "product_id",
    "title",
    "source",
    "price",
    "currency",
    "product_average_rating",
    "metadata_rating_count",
    "product_url",
    "features"
]


field_isolation_results = {}


for field in stable_fields:

    before = normalize_value(
        context_ke_off.get(field)
    )

    after = normalize_value(
        context_ke_on.get(field)
    )

    unchanged = (
        before == after
    )

    field_isolation_results[field] = (
        unchanged
    )

    print(
        f"{field:25} ->",
        "UNCHANGED"
        if unchanged
        else "CHANGED"
    )


description_changed = (
    context_ke_off["description"]
    != context_ke_on["description"]
)


field_isolation_pass = (
    all(
        field_isolation_results.values()
    )
    and description_changed
)


print(
    "\nDESCRIPTION CHANGED:",
    description_changed
)

print(
    "FIELD ISOLATION PASS:",
    field_isolation_pass
)

assert field_isolation_pass


print(
    "\nPASS: ONLY TARGET FIELD CHANGED"
)

KNOWLEDGE EDIT — FIELD ISOLATION TEST
product_id                -> UNCHANGED
title                     -> UNCHANGED
source                    -> UNCHANGED
price                     -> UNCHANGED
currency                  -> UNCHANGED
product_average_rating    -> UNCHANGED
metadata_rating_count     -> UNCHANGED
product_url               -> UNCHANGED
features                  -> UNCHANGED

DESCRIPTION CHANGED: True
FIELD ISOLATION PASS: True

PASS: ONLY TARGET FIELD CHANGED


# Kiểm tra Review Evidence không bị Knowledge Editing làm thay đổi

In [30]:
# ============================================================
# CELL 25 — REVIEW EVIDENCE INTEGRITY TEST
# ============================================================

product_reviews = reviews[
    reviews["product_id"].astype(str)
    == TEST_PRODUCT_ID
].copy()


review_ids_before = set(
    product_reviews[
        "review_id"
    ].astype(str)
)


# Gọi lại effective knowledge
_ = get_effective_product_knowledge(
    TEST_PRODUCT_ID
)


product_reviews_after = reviews[
    reviews["product_id"].astype(str)
    == TEST_PRODUCT_ID
].copy()


review_ids_after = set(
    product_reviews_after[
        "review_id"
    ].astype(str)
)


review_integrity_pass = (
    review_ids_before
    == review_ids_after
)


print("=" * 80)
print("REVIEW EVIDENCE INTEGRITY")
print("=" * 80)

print(
    "PRODUCT:",
    TEST_PRODUCT_ID
)

print(
    "REVIEWS BEFORE:",
    len(review_ids_before)
)

print(
    "REVIEWS AFTER:",
    len(review_ids_after)
)

print(
    "REVIEW IDS IDENTICAL:",
    review_integrity_pass
)

assert review_integrity_pass


print(
    "\nPASS: KNOWLEDGE EDIT "
    "DID NOT MODIFY REVIEW EVIDENCE"
)

REVIEW EVIDENCE INTEGRITY
PRODUCT: amazon_B09HKJY2QB
REVIEWS BEFORE: 33
REVIEWS AFTER: 33
REVIEW IDS IDENTICAL: True

PASS: KNOWLEDGE EDIT DID NOT MODIFY REVIEW EVIDENCE


# Tạo text embedding cho Effective Knowledge

In [31]:
# ============================================================
# CELL 26 — BUILD KE EMBEDDING TEXT
# ============================================================

def clean_text_value(value):

    value = normalize_value(value)

    if value is None:
        return ""

    return str(value).strip()


def build_ke_embedding_text(product_id):

    result = get_effective_product_knowledge(
        product_id
    )

    p = result["effective"]

    parts = []

    fields = [
        ("Product", p.get("product_name")),
        ("Category", p.get("category")),
        ("Subcategory", p.get("subcategory")),
        ("Brand", p.get("brand")),
        ("Description", p.get("description")),
        ("Features", p.get("product_features")),
    ]

    for label, value in fields:

        text = clean_text_value(value)

        if text:
            parts.append(
                f"{label}: {text}"
            )

    return "\n".join(parts)


ke_embedding_text = (
    build_ke_embedding_text(
        TEST_PRODUCT_ID
    )
)


print("=" * 80)
print("KE EMBEDDING TEXT")
print("=" * 80)

print(
    ke_embedding_text[:3000]
)

print(
    "\nTEXT LENGTH:",
    len(ke_embedding_text)
)

assert TEST_DESCRIPTION in (
    ke_embedding_text
)

print(
    "\nPASS: ACTIVE KNOWLEDGE EDIT "
    "IS PRESENT IN EMBEDDING TEXT"
)

KE EMBEDDING TEXT
Product: Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
Category: All Electronics
Brand: INLAND
Description: [CONTROLLED KE TEST] This description was added by the Step 14 Knowledge Editing evaluation layer.
Features: Micro SD Card 64GB. Fast Read/Write Speed up to: 95MB/s and 30MB/s respectively for high resolution photo capturing. Extended Capacity for pictures, music, documents
4K UHD Capable and Full HD Ready with UHS speed class 3 (U3) and video speed class 30 (V30). Cool travel gadgets for action cameras, DSLR, drones, laptops and more
Rated Application Class 1 (A1) for faster app loading and enhanced app performance. Great storage accessories for tablets, smartphones, games consoles, android devices. A compatible memory card for Nintendo Switch and Switch Lite, a system update is required for using a microSDXC card, visit the Nintendo Switch official website fo

# Tạo Edit Vector Overlay

In [32]:
# ============================================================
# CELL 27 — BUILD KNOWLEDGE EDIT VECTOR OVERLAY
# ============================================================

def get_active_edited_product_ids():

    edits = load_knowledge_edits()

    return sorted({
        str(edit["product_id"])
        for edit in edits
        if edit.get("status") == "active"
    })


def build_ke_vector_overlay():

    edited_product_ids = (
        get_active_edited_product_ids()
    )

    overlay = {}

    if not edited_product_ids:
        return overlay

    texts = [
        build_ke_embedding_text(pid)
        for pid in edited_product_ids
    ]

    vectors = product_query_model.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    for product_id, text, vector in zip(
        edited_product_ids,
        texts,
        vectors
    ):

        overlay[product_id] = {
            "product_id": product_id,
            "embedding_text": text,
            "vector": vector
        }

    return overlay


ke_vector_overlay = (
    build_ke_vector_overlay()
)


print("=" * 80)
print("KNOWLEDGE EDIT VECTOR OVERLAY")
print("=" * 80)

print(
    "ACTIVE EDITED PRODUCTS:",
    len(
        get_active_edited_product_ids()
    )
)

print(
    "OVERLAY VECTOR COUNT:",
    len(ke_vector_overlay)
)


for pid, data in (
    ke_vector_overlay.items()
):

    print("\nPRODUCT:", pid)

    print(
        "VECTOR SHAPE:",
        data["vector"].shape
    )

    print(
        "VECTOR NORM:",
        round(
            float(
                np.linalg.norm(
                    data["vector"]
                )
            ),
            6
        )
    )


assert (
    TEST_PRODUCT_ID
    in ke_vector_overlay
)

assert (
    ke_vector_overlay[
        TEST_PRODUCT_ID
    ]["vector"].shape[0]
    == faiss_index.d
)


print(
    "\nPASS: KE VECTOR OVERLAY READY"
)

KNOWLEDGE EDIT VECTOR OVERLAY
ACTIVE EDITED PRODUCTS: 1
OVERLAY VECTOR COUNT: 1

PRODUCT: amazon_B09HKJY2QB
VECTOR SHAPE: (384,)
VECTOR NORM: 1.0

PASS: KE VECTOR OVERLAY READY


# So sánh Original Vector và Edited Vector

In [33]:
# ============================================================
# CELL 28 — ORIGINAL vs EDITED VECTOR TEST
# ============================================================

meta_match = embedding_metadata[
    embedding_metadata[
        "product_id"
    ].astype(str)
    == TEST_PRODUCT_ID
]

assert len(meta_match) == 1


original_faiss_row = int(
    meta_match.iloc[0][
        "faiss_row_id"
    ]
)


original_vector = (
    faiss_index.reconstruct(
        original_faiss_row
    )
)

edited_vector = (
    ke_vector_overlay[
        TEST_PRODUCT_ID
    ]["vector"]
)


original_norm = float(
    np.linalg.norm(
        original_vector
    )
)

edited_norm = float(
    np.linalg.norm(
        edited_vector
    )
)


vector_similarity = float(
    np.dot(
        original_vector,
        edited_vector
    )
    /
    (
        original_norm
        * edited_norm
    )
)


vector_changed = (
    not np.allclose(
        original_vector,
        edited_vector,
        atol=1e-6
    )
)


print("=" * 80)
print("ORIGINAL vs EDITED EMBEDDING")
print("=" * 80)

print(
    "PRODUCT:",
    TEST_PRODUCT_ID
)

print(
    "FAISS ROW:",
    original_faiss_row
)

print(
    "ORIGINAL NORM:",
    round(
        original_norm,
        6
    )
)

print(
    "EDITED NORM:",
    round(
        edited_norm,
        6
    )
)

print(
    "COSINE SIMILARITY:",
    round(
        vector_similarity,
        6
    )
)

print(
    "VECTOR CHANGED:",
    vector_changed
)

assert vector_changed

print(
    "\nPASS: KNOWLEDGE EDIT "
    "PRODUCED A NEW REPRESENTATION"
)


ORIGINAL vs EDITED EMBEDDING
PRODUCT: amazon_B09HKJY2QB
FAISS ROW: 1254
ORIGINAL NORM: 1.0
EDITED NORM: 1.0
COSINE SIMILARITY: 0.927488
VECTOR CHANGED: True

PASS: KNOWLEDGE EDIT PRODUCED A NEW REPRESENTATION


# Hàm KE-aware Semantic Retrieval

In [34]:
# ============================================================
# CELL 29 — KE-AWARE SEMANTIC RETRIEVAL
# ============================================================

def ke_aware_semantic_search(
    query,
    top_k=10,
    faiss_k=200
):

    query_vector = (
        product_query_model.encode(
            [query],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )
        .astype("float32")
    )

    # --------------------------------
    # 1. Base FAISS retrieval
    # --------------------------------

    scores, indices = (
        faiss_index.search(
            query_vector,
            faiss_k
        )
    )

    candidate_scores = {}

    for score, row_id in zip(
        scores[0],
        indices[0]
    ):

        row_id = int(row_id)

        if row_id < 0:
            continue

        meta = (
            metadata_by_faiss_row
            .loc[row_id]
        )

        product_id = str(
            meta["product_id"]
        )

        candidate_scores[
            product_id
        ] = {
            "product_id": product_id,
            "semantic_score": float(score),
            "score_source": "base_faiss",
            "faiss_row_id": row_id
        }

    # --------------------------------
    # 2. KE vector overlay
    # --------------------------------

    q = query_vector[0]

    for product_id, data in (
        ke_vector_overlay.items()
    ):

        ke_score = float(
            np.dot(
                q,
                data["vector"]
            )
        )

        if product_id in candidate_scores:

            candidate_scores[
                product_id
            ][
                "base_semantic_score"
            ] = (
                candidate_scores[
                    product_id
                ][
                    "semantic_score"
                ]
            )

        else:

            candidate_scores[
                product_id
            ] = {
                "product_id": product_id,
                "faiss_row_id": None
            }

        candidate_scores[
            product_id
        ][
            "semantic_score"
        ] = ke_score

        candidate_scores[
            product_id
        ][
            "score_source"
        ] = "ke_overlay"

    # --------------------------------
    # 3. Sort
    # --------------------------------

    ranked = sorted(
        candidate_scores.values(),
        key=lambda x: x[
            "semantic_score"
        ],
        reverse=True
    )

    ranked = ranked[:top_k]

    # --------------------------------
    # 4. Add effective metadata
    # --------------------------------

    results = []

    for rank, item in enumerate(
        ranked,
        start=1
    ):

        effective = (
            get_effective_product_knowledge(
                item["product_id"]
            )["effective"]
        )

        row = dict(item)

        row["rank"] = rank

        row["title"] = (
            effective.get(
                "product_name"
            )
        )

        row["price"] = (
            effective.get("price")
        )

        row["currency"] = (
            effective.get(
                "currency"
            )
        )

        results.append(row)

    return results


print(
    "PASS: KE-AWARE RETRIEVAL READY"
)

PASS: KE-AWARE RETRIEVAL READY


# Tạo semantic test edit

In [35]:
# ============================================================
# CELL 30 — CREATE CONTROLLED SEMANTIC EDIT
# ============================================================

SEMANTIC_TEST_MARKER = (
    "KESEMANTICZXQ9274"
)

SEMANTIC_TEST_DESCRIPTION = (
    "[CONTROLLED KE RETRIEVAL TEST] "
    "KESEMANTICZXQ9274 semantic retrieval marker."
)


semantic_test_edit = create_knowledge_edit(
    product_id=TEST_PRODUCT_ID,

    field="description",

    new_value=SEMANTIC_TEST_DESCRIPTION,

    reason=(
        "Controlled experiment for measuring "
        "retrieval behavior after knowledge editing."
    ),

    evidence={
        "type": "controlled_retrieval_experiment",
        "verified_product_claim": False,
        "marker": SEMANTIC_TEST_MARKER
    },

    provenance="controlled_test",

    activate=False
)


print("=" * 80)
print("CONTROLLED SEMANTIC EDIT")
print("=" * 80)

print(
    json.dumps(
        semantic_test_edit,
        ensure_ascii=False,
        indent=2,
        default=str
    )
)

print(
    "\nPASS: SEMANTIC TEST EDIT CREATED"
)

CONTROLLED SEMANTIC EDIT
{
  "edit_id": "7221b1e1-d118-4d93-b361-081328e5c245",
  "product_id": "amazon_B09HKJY2QB",
  "field": "description",
  "old_value": null,
  "new_value": "[CONTROLLED KE RETRIEVAL TEST] KESEMANTICZXQ9274 semantic retrieval marker.",
  "reason": "Controlled experiment for measuring retrieval behavior after knowledge editing.",
  "evidence": {
    "type": "controlled_retrieval_experiment",
    "verified_product_claim": false,
    "marker": "KESEMANTICZXQ9274"
  },
  "provenance": "controlled_test",
  "created_at": "2026-09-29T07:25:55.117605+00:00",
  "status": "inactive",
  "version": 2
}

PASS: SEMANTIC TEST EDIT CREATED


In [36]:
# ============================================================
# CELL 31 — COMMIT SEMANTIC TEST EDIT
# ============================================================

knowledge_edits = (
    load_knowledge_edits()
)

existing_ids = {
    x["edit_id"]
    for x in knowledge_edits
}


if (
    semantic_test_edit["edit_id"]
    not in existing_ids
):

    knowledge_edits.append(
        semantic_test_edit
    )

    save_knowledge_edits(
        knowledge_edits
    )


print("=" * 80)
print("SEMANTIC EDIT COMMITTED")
print("=" * 80)

print(
    "EDIT ID:",
    semantic_test_edit["edit_id"]
)

print(
    "VERSION:",
    semantic_test_edit["version"]
)

print(
    "STATUS:",
    semantic_test_edit["status"]
)

print(
    "TOTAL EDITS:",
    len(load_knowledge_edits())
)

print(
    "\nPASS: SEMANTIC TEST EDIT STORED"
)

SEMANTIC EDIT COMMITTED
EDIT ID: 7221b1e1-d118-4d93-b361-081328e5c245
VERSION: 2
STATUS: inactive
TOTAL EDITS: 2

PASS: SEMANTIC TEST EDIT STORED


In [37]:
# ============================================================
# CELL 32 — RETRIEVAL BEFORE SEMANTIC EDIT
# ============================================================

SEMANTIC_TEST_QUERY = (
    "KESEMANTICZXQ9274"
)


# Rebuild overlay ở trạng thái BEFORE
ke_vector_overlay = (
    build_ke_vector_overlay()
)


def find_product_rank(
    query,
    target_product_id,
    faiss_k=1000
):

    results = (
        ke_aware_semantic_search(
            query=query,
            top_k=faiss_k + 100,
            faiss_k=faiss_k
        )
    )

    for row in results:

        if (
            str(row["product_id"])
            == str(target_product_id)
        ):

            return {
                "rank": row["rank"],
                "semantic_score": (
                    row["semantic_score"]
                ),
                "score_source": (
                    row["score_source"]
                )
            }

    return {
        "rank": None,
        "semantic_score": None,
        "score_source": None
    }


before_semantic_edit = (
    find_product_rank(
        SEMANTIC_TEST_QUERY,
        TEST_PRODUCT_ID,
        faiss_k=1000
    )
)


print("=" * 80)
print("BEFORE SEMANTIC EDIT")
print("=" * 80)

print(
    "QUERY:",
    SEMANTIC_TEST_QUERY
)

print(
    "TARGET:",
    TEST_PRODUCT_ID
)

print(
    "RANK:",
    before_semantic_edit["rank"]
)

print(
    "SCORE:",
    before_semantic_edit[
        "semantic_score"
    ]
)

print(
    "SOURCE:",
    before_semantic_edit[
        "score_source"
    ]
)

BEFORE SEMANTIC EDIT
QUERY: KESEMANTICZXQ9274
TARGET: amazon_B09HKJY2QB
RANK: 1001
SCORE: 0.10230742394924164
SOURCE: ke_overlay


In [38]:
# ============================================================
# CELL 33 — EXCLUSIVE FIELD VERSION ACTIVATION
# ============================================================

def activate_edit_exclusive(edit_id):

    edits = load_knowledge_edits()

    target = None

    for edit in edits:

        if edit.get("edit_id") == edit_id:
            target = edit
            break

    if target is None:
        raise ValueError(
            f"Edit not found: {edit_id}"
        )

    target_product = str(
        target["product_id"]
    )

    target_field = (
        target["field"]
    )

    now = datetime.now(
        timezone.utc
    ).isoformat()


    for edit in edits:

        same_scope = (
            str(edit.get("product_id"))
            == target_product
            and
            edit.get("field")
            == target_field
        )

        if same_scope:

            if (
                edit.get("edit_id")
                == edit_id
            ):
                edit["status"] = "active"
            else:
                edit["status"] = "inactive"

            edit[
                "status_updated_at"
            ] = now


    save_knowledge_edits(
        edits
    )

    return True


activate_edit_exclusive(
    semantic_test_edit["edit_id"]
)


active_description_edits = [
    e
    for e in load_knowledge_edits()
    if (
        str(e.get("product_id"))
        == TEST_PRODUCT_ID
        and e.get("field")
        == "description"
        and e.get("status")
        == "active"
    )
]


print("=" * 80)
print("ACTIVE DESCRIPTION VERSION")
print("=" * 80)

for edit in active_description_edits:

    print(
        "VERSION:",
        edit["version"]
    )

    print(
        "EDIT ID:",
        edit["edit_id"]
    )

    print(
        "VALUE:",
        edit["new_value"]
    )


assert (
    len(active_description_edits)
    == 1
)

assert (
    active_description_edits[0][
        "edit_id"
    ]
    == semantic_test_edit["edit_id"]
)

print(
    "\nPASS: ONLY VERSION 2 ACTIVE"
)

ACTIVE DESCRIPTION VERSION
VERSION: 2
EDIT ID: 7221b1e1-d118-4d93-b361-081328e5c245
VALUE: [CONTROLLED KE RETRIEVAL TEST] KESEMANTICZXQ9274 semantic retrieval marker.

PASS: ONLY VERSION 2 ACTIVE


In [39]:
# ============================================================
# CELL 34 — RETRIEVAL AFTER SEMANTIC EDIT
# ============================================================

ke_vector_overlay = (
    build_ke_vector_overlay()
)


effective_after_semantic_edit = (
    get_effective_product_knowledge(
        TEST_PRODUCT_ID
    )
)


print("=" * 80)
print("EFFECTIVE KNOWLEDGE")
print("=" * 80)

print(
    "DESCRIPTION:"
)

print(
    effective_after_semantic_edit[
        "effective"
    ]["description"]
)


after_semantic_edit = (
    find_product_rank(
        SEMANTIC_TEST_QUERY,
        TEST_PRODUCT_ID,
        faiss_k=1000
    )
)


print("\n" + "=" * 80)
print("AFTER SEMANTIC EDIT")
print("=" * 80)

print(
    "RANK:",
    after_semantic_edit["rank"]
)

print(
    "SCORE:",
    after_semantic_edit[
        "semantic_score"
    ]
)

print(
    "SOURCE:",
    after_semantic_edit[
        "score_source"
    ]
)

EFFECTIVE KNOWLEDGE
DESCRIPTION:
[CONTROLLED KE RETRIEVAL TEST] KESEMANTICZXQ9274 semantic retrieval marker.

AFTER SEMANTIC EDIT
RANK: 1001
SCORE: 0.1669391393661499
SOURCE: ke_overlay


In [41]:
# ============================================================
# CELL 36 — CLEAN UP FAILED MARKER EXPERIMENT
# ============================================================

# Deactivate version 2
deactivate_edit(
    semantic_test_edit["edit_id"]
)

# Reactivate version 1 exclusively
activate_edit_exclusive(
    first_edit["edit_id"]
)


current_effective = (
    get_effective_product_knowledge(
        TEST_PRODUCT_ID
    )
)


active_description_edits = [
    e
    for e in load_knowledge_edits()
    if (
        str(e.get("product_id"))
        == TEST_PRODUCT_ID
        and
        e.get("field")
        == "description"
        and
        e.get("status")
        == "active"
    )
]


print("=" * 80)
print("SEMANTIC TEST CLEANUP")
print("=" * 80)

print(
    "ACTIVE DESCRIPTION EDITS:",
    len(active_description_edits)
)

print(
    "ACTIVE VERSION:",
    active_description_edits[0]["version"]
)

print(
    "\nEFFECTIVE DESCRIPTION:"
)

print(
    current_effective[
        "effective"
    ]["description"]
)


assert (
    len(active_description_edits)
    == 1
)

assert (
    active_description_edits[0][
        "edit_id"
    ]
    == first_edit["edit_id"]
)

print(
    "\nPASS: MARKER EXPERIMENT ROLLED BACK"
)

SEMANTIC TEST CLEANUP
ACTIVE DESCRIPTION EDITS: 1
ACTIVE VERSION: 1

EFFECTIVE DESCRIPTION:
[CONTROLLED KE TEST] This description was added by the Step 14 Knowledge Editing evaluation layer.

PASS: MARKER EXPERIMENT ROLLED BACK


In [42]:
# ============================================================
# CELL 37 — CREATE MEANINGFUL SEMANTIC TEST EDIT
# ============================================================

SEMANTIC_TEST_QUERY_V2 = (
    "memory card for underwater photography"
)

SEMANTIC_TEST_DESCRIPTION_V2 = (
    "[CONTROLLED RETRIEVAL TEST — NOT VERIFIED PRODUCT FACT] "
    "Memory card intended for underwater photography."
)


semantic_edit_v2 = create_knowledge_edit(
    product_id=TEST_PRODUCT_ID,

    field="description",

    new_value=SEMANTIC_TEST_DESCRIPTION_V2,

    reason=(
        "Controlled semantic retrieval evaluation. "
        "This value is not treated as a verified "
        "product fact."
    ),

    evidence={
        "type": "controlled_semantic_evaluation",
        "verified_product_claim": False,
        "test_query": SEMANTIC_TEST_QUERY_V2
    },

    provenance="controlled_test",

    activate=False
)


# Commit
edits = load_knowledge_edits()

if not any(
    e["edit_id"]
    == semantic_edit_v2["edit_id"]
    for e in edits
):

    edits.append(
        semantic_edit_v2
    )

    save_knowledge_edits(
        edits
    )


print("=" * 80)
print("SEMANTIC TEST V2")
print("=" * 80)

print(
    "QUERY:",
    SEMANTIC_TEST_QUERY_V2
)

print(
    "\nNEW DESCRIPTION:"
)

print(
    SEMANTIC_TEST_DESCRIPTION_V2
)

print(
    "\nEDIT VERSION:",
    semantic_edit_v2["version"]
)

print(
    "STATUS:",
    semantic_edit_v2["status"]
)

print(
    "\nPASS: CONTROLLED SEMANTIC EDIT READY"
)

SEMANTIC TEST V2
QUERY: memory card for underwater photography

NEW DESCRIPTION:
[CONTROLLED RETRIEVAL TEST — NOT VERIFIED PRODUCT FACT] Memory card intended for underwater photography.

EDIT VERSION: 3
STATUS: inactive

PASS: CONTROLLED SEMANTIC EDIT READY


In [43]:
# ============================================================
# CELL 38 — MEASURE BASE REPRESENTATION
# ============================================================

query_v2_vector = (
    product_query_model.encode(
        [SEMANTIC_TEST_QUERY_V2],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )
    .astype("float32")[0]
)


# Vector gốc từ Step 11
original_vector = (
    faiss_index.reconstruct(
        original_faiss_row
    )
)


base_similarity = float(
    np.dot(
        query_v2_vector,
        original_vector
    )
)


print("=" * 80)
print("BEFORE KNOWLEDGE EDIT")
print("=" * 80)

print(
    "QUERY:",
    SEMANTIC_TEST_QUERY_V2
)

print(
    "TARGET:",
    TEST_PRODUCT_ID
)

print(
    "BASE VECTOR SIMILARITY:",
    round(
        base_similarity,
        6
    )
)

BEFORE KNOWLEDGE EDIT
QUERY: memory card for underwater photography
TARGET: amazon_B09HKJY2QB
BASE VECTOR SIMILARITY: 0.456486


In [44]:
# ============================================================
# CELL 39 — ACTIVATE SEMANTIC EDIT V2
# ============================================================

activate_edit_exclusive(
    semantic_edit_v2["edit_id"]
)


ke_vector_overlay = (
    build_ke_vector_overlay()
)


assert (
    TEST_PRODUCT_ID
    in ke_vector_overlay
)


edited_v2_vector = (
    ke_vector_overlay[
        TEST_PRODUCT_ID
    ]["vector"]
)


effective_v2 = (
    get_effective_product_knowledge(
        TEST_PRODUCT_ID
    )
)


print("=" * 80)
print("SEMANTIC EDIT V2 ACTIVE")
print("=" * 80)

print(
    "EFFECTIVE DESCRIPTION:"
)

print(
    effective_v2[
        "effective"
    ]["description"]
)

print(
    "\nVECTOR NORM:",
    round(
        float(
            np.linalg.norm(
                edited_v2_vector
            )
        ),
        6
    )
)

print(
    "\nPASS: EDITED VECTOR READY"
)

SEMANTIC EDIT V2 ACTIVE
EFFECTIVE DESCRIPTION:
[CONTROLLED RETRIEVAL TEST — NOT VERIFIED PRODUCT FACT] Memory card intended for underwater photography.

VECTOR NORM: 1.0

PASS: EDITED VECTOR READY


In [45]:
# ============================================================
# CELL 40 — SEMANTIC EDIT EFFECT TEST
# ============================================================

edited_similarity = float(
    np.dot(
        query_v2_vector,
        edited_v2_vector
    )
)


similarity_delta = (
    edited_similarity
    - base_similarity
)


if abs(base_similarity) > 1e-12:

    relative_change_pct = (
        similarity_delta
        / abs(base_similarity)
        * 100
    )

else:

    relative_change_pct = None


semantic_effect_pass = (
    edited_similarity
    > base_similarity
)


print("=" * 80)
print("KNOWLEDGE EDIT — SEMANTIC EFFECT")
print("=" * 80)

print(
    "QUERY:",
    SEMANTIC_TEST_QUERY_V2
)

print(
    "\nBEFORE:",
    round(
        base_similarity,
        6
    )
)

print(
    "AFTER:",
    round(
        edited_similarity,
        6
    )
)

print(
    "DELTA:",
    round(
        similarity_delta,
        6
    )
)


if relative_change_pct is not None:

    print(
        "RELATIVE CHANGE:",
        f"{relative_change_pct:.2f}%"
    )


print(
    "\nSIMILARITY IMPROVED:",
    semantic_effect_pass
)


assert semantic_effect_pass


print(
    "\nPASS: EDITED KNOWLEDGE "
    "INCREASED QUERY RELEVANCE"
)

KNOWLEDGE EDIT — SEMANTIC EFFECT
QUERY: memory card for underwater photography

BEFORE: 0.456486
AFTER: 0.684967
DELTA: 0.228482
RELATIVE CHANGE: 50.05%

SIMILARITY IMPROVED: True

PASS: EDITED KNOWLEDGE INCREASED QUERY RELEVANCE


In [46]:
# ============================================================
# CELL 41 — OBSERVE RETRIEVAL RANK
# ============================================================

ke_vector_overlay = (
    build_ke_vector_overlay()
)


rank_after_v2 = (
    find_product_rank(
        SEMANTIC_TEST_QUERY_V2,
        TEST_PRODUCT_ID,
        faiss_k=1000
    )
)


print("=" * 80)
print("KE-AWARE RETRIEVAL OBSERVATION")
print("=" * 80)

print(
    "QUERY:",
    SEMANTIC_TEST_QUERY_V2
)

print(
    "TARGET:",
    TEST_PRODUCT_ID
)

print(
    "\nRANK:",
    rank_after_v2["rank"]
)

print(
    "SCORE:",
    rank_after_v2[
        "semantic_score"
    ]
)

print(
    "SOURCE:",
    rank_after_v2[
        "score_source"
    ]
)


if (
    rank_after_v2["rank"]
    is not None
):

    if rank_after_v2["rank"] <= 1000:

        print(
            "\nTARGET ENTERED TOP-1000 "
            "RETRIEVAL RESULTS"
        )

    else:

        print(
            "\nTarget did not enter "
            "top-1000."
        )


print(
    "\nNOTE: Rank is an observational "
    "metric; semantic similarity is the "
    "primary controlled-test metric."
)

KE-AWARE RETRIEVAL OBSERVATION
QUERY: memory card for underwater photography
TARGET: amazon_B09HKJY2QB

RANK: 3
SCORE: 0.6849671602249146
SOURCE: ke_overlay

TARGET ENTERED TOP-1000 RETRIEVAL RESULTS

NOTE: Rank is an observational metric; semantic similarity is the primary controlled-test metric.


In [47]:
# ============================================================
# CELL 42 — CLEAN CONTROLLED TEST EDITS
# ============================================================

edits = load_knowledge_edits()

deactivated_count = 0

for edit in edits:

    if (
        str(edit.get("product_id"))
        == TEST_PRODUCT_ID
        and
        edit.get("provenance")
        == "controlled_test"
    ):

        if edit.get("status") == "active":

            edit["status"] = "inactive"

            edit["status_updated_at"] = (
                datetime.now(
                    timezone.utc
                ).isoformat()
            )

            deactivated_count += 1


save_knowledge_edits(edits)


# Rebuild overlay
ke_vector_overlay = (
    build_ke_vector_overlay()
)


effective_clean = (
    get_effective_product_knowledge(
        TEST_PRODUCT_ID
    )
)


print("=" * 80)
print("CONTROLLED TEST CLEANUP")
print("=" * 80)

print(
    "DEACTIVATED:",
    deactivated_count
)

print(
    "ACTIVE EDIT COUNT:",
    effective_clean[
        "active_edit_count"
    ]
)

print(
    "EFFECTIVE DESCRIPTION:"
)

print(
    effective_clean[
        "effective"
    ].get("description")
)

print(
    "\nKE OVERLAY PRODUCTS:",
    len(ke_vector_overlay)
)


cleanup_pass = (
    effective_clean[
        "active_edit_count"
    ] == 0
    and
    effective_clean[
        "effective"
    ].get("description")
    is None
    and
    TEST_PRODUCT_ID
    not in ke_vector_overlay
)


print(
    "\nCLEANUP PASS:",
    cleanup_pass
)

assert cleanup_pass

print(
    "\nPASS: CONTROLLED KNOWLEDGE "
    "REMOVED FROM ACTIVE SYSTEM"
)

CONTROLLED TEST CLEANUP
DEACTIVATED: 1
ACTIVE EDIT COUNT: 0
EFFECTIVE DESCRIPTION:
None

KE OVERLAY PRODUCTS: 0

CLEANUP PASS: True

PASS: CONTROLLED KNOWLEDGE REMOVED FROM ACTIVE SYSTEM


In [48]:
# ============================================================
# CELL 43 — SAVE STEP 14 KE EVALUATION
# ============================================================

semantic_evaluation = {
    "test_type": (
        "controlled_semantic_knowledge_edit"
    ),

    "product_id": TEST_PRODUCT_ID,

    "query": SEMANTIC_TEST_QUERY_V2,

    "verified_product_claim": False,

    "before_similarity": float(
        base_similarity
    ),

    "after_similarity": float(
        edited_similarity
    ),

    "similarity_delta": float(
        similarity_delta
    ),

    "relative_change_percent": float(
        relative_change_pct
    ),

    "retrieval_rank_after_edit": (
        rank_after_v2["rank"]
    ),

    "retrieval_score_after_edit": (
        rank_after_v2[
            "semantic_score"
        ]
    ),

    "retrieval_source": (
        rank_after_v2[
            "score_source"
        ]
    ),

    "similarity_improved": bool(
        semantic_effect_pass
    )
}


step14_evaluation = {
    "step": "step14_knowledge_editing",

    "method": (
        "retrieval_time_knowledge_edit_overlay"
    ),

    "parameter_weight_editing": False,

    "tests": {
        "reliability": True,
        "locality": True,
        "persistence": True,
        "rollback": True,
        "reactivation": True,
        "field_isolation": True,
        "review_integrity": True,
        "rag_context_update": True,
        "embedding_update": True,
        "semantic_effect": (
            semantic_evaluation
        )
    },

    "controlled_test_edits_active": False,

    "notes": [
        (
            "Knowledge edits are stored "
            "separately from the source dataset."
        ),
        (
            "Controlled semantic edits are "
            "evaluation artifacts and are not "
            "verified product facts."
        ),
        (
            "Edited products use a vector "
            "overlay instead of rebuilding "
            "the full FAISS index."
        )
    ]
}


STEP14_EVAL_PATH = (
    STEP14_OUTPUT_DIR
    / "step14_ke_evaluation.json"
)


with open(
    STEP14_EVAL_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step14_evaluation,
        f,
        ensure_ascii=False,
        indent=2
    )


print("=" * 80)
print("STEP 14 EVALUATION SAVED")
print("=" * 80)

print(
    json.dumps(
        semantic_evaluation,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "\nSAVED:"
)

print(
    STEP14_EVAL_PATH
)

print(
    "\nPASS: KE EVALUATION SAVED"
)

STEP 14 EVALUATION SAVED
{
  "test_type": "controlled_semantic_knowledge_edit",
  "product_id": "amazon_B09HKJY2QB",
  "query": "memory card for underwater photography",
  "verified_product_claim": false,
  "before_similarity": 0.4564856290817261,
  "after_similarity": 0.6849671602249146,
  "similarity_delta": 0.22848153114318848,
  "relative_change_percent": 50.052294439762676,
  "retrieval_rank_after_edit": 3,
  "retrieval_score_after_edit": 0.6849671602249146,
  "retrieval_source": "ke_overlay",
  "similarity_improved": true
}

SAVED:
/kaggle/working/step14/step14_ke_evaluation.json

PASS: KE EVALUATION SAVED


In [49]:
# ============================================================
# CELL 44 — TRUSTED KNOWLEDGE EDIT API
# ============================================================

def propose_product_knowledge_edit(
    product_id,
    field,
    new_value,
    reason,
    evidence,
    provenance
):

    if not reason:
        raise ValueError(
            "reason is required"
        )

    if not evidence:
        raise ValueError(
            "evidence is required"
        )

    if not provenance:
        raise ValueError(
            "provenance is required"
        )

    edit = create_knowledge_edit(
        product_id=product_id,
        field=field,
        new_value=new_value,
        reason=reason,
        evidence=evidence,
        provenance=provenance,
        activate=False
    )

    return edit


print("=" * 80)
print("TRUSTED KNOWLEDGE EDIT API")
print("=" * 80)

print(
    "Required:"
)

print(
    "- product_id"
)

print(
    "- field"
)

print(
    "- new_value"
)

print(
    "- reason"
)

print(
    "- evidence"
)

print(
    "- provenance"
)

print(
    "\nNew edits default to INACTIVE "
    "until explicitly approved."
)

print(
    "\nPASS: TRUSTED EDIT API READY"
)

TRUSTED KNOWLEDGE EDIT API
Required:
- product_id
- field
- new_value
- reason
- evidence
- provenance

New edits default to INACTIVE until explicitly approved.

PASS: TRUSTED EDIT API READY


In [50]:
# ============================================================
# CELL 45 — QUERY REQUIREMENT PARSER
# ============================================================

import re


def parse_ke_query_requirements(query):

    text = str(query).strip()

    result = {
        "original_query": text,
        "capacity_gb": None,
        "max_price": None,
        "currency": None
    }

    # ----------------------------
    # Capacity
    # ----------------------------

    capacity_match = re.search(
        r"\b(\d+(?:\.\d+)?)\s*(GB|TB)\b",
        text,
        flags=re.I
    )

    if capacity_match:

        value = float(
            capacity_match.group(1)
        )

        unit = (
            capacity_match
            .group(2)
            .upper()
        )

        if unit == "TB":
            value *= 1024

        result["capacity_gb"] = (
            int(value)
            if value.is_integer()
            else value
        )

    # ----------------------------
    # Budget
    # ----------------------------

    price_patterns = [
        r"(?:under|below|less than|max(?:imum)?|<=?)\s*\$?\s*(\d+(?:\.\d+)?)",
        r"\$?\s*(\d+(?:\.\d+)?)\s*(?:usd|dollars?)"
    ]

    for pattern in price_patterns:

        match = re.search(
            pattern,
            text,
            flags=re.I
        )

        if match:
            result["max_price"] = float(
                match.group(1)
            )
            break

    # ----------------------------
    # Currency
    # ----------------------------

    if re.search(
        r"\bUSD\b|\$",
        text,
        flags=re.I
    ):
        result["currency"] = "USD"

    elif re.search(
        r"\bVND\b|₫|đồng",
        text,
        flags=re.I
    ):
        result["currency"] = "VND"

    return result


test_req = parse_ke_query_requirements(
    "microSD 64GB under 20 USD"
)

print(test_req)

assert test_req["capacity_gb"] == 64
assert test_req["max_price"] == 20.0
assert test_req["currency"] == "USD"

print(
    "\nPASS: QUERY REQUIREMENT PARSER READY"
)

{'original_query': 'microSD 64GB under 20 USD', 'capacity_gb': 64, 'max_price': 20.0, 'currency': 'USD'}

PASS: QUERY REQUIREMENT PARSER READY


In [51]:
# ============================================================
# CELL 46 — CAPACITY EXTRACTION
# ============================================================

def extract_capacities_gb(text):

    text = clean_text_value(text)

    capacities = []

    for value, unit in re.findall(
        r"(?<!\d)(\d+(?:\.\d+)?)\s*(GB|TB)\b",
        text,
        flags=re.I
    ):

        value = float(value)

        if unit.upper() == "TB":
            value *= 1024

        capacities.append(
            int(value)
            if value.is_integer()
            else value
        )

    return capacities


tests = [
    "microSD 64GB",
    "2-Pack 64GB microSD",
    "1TB memory card"
]


for t in tests:

    print(
        t,
        "->",
        extract_capacities_gb(t)
    )


print(
    "\nPASS: CAPACITY EXTRACTOR READY"
)

microSD 64GB -> [64]
2-Pack 64GB microSD -> [64]
1TB memory card -> [1024]

PASS: CAPACITY EXTRACTOR READY


In [52]:
# ============================================================
# CELL 47 — KE-AWARE CANDIDATE RETRIEVAL
# ============================================================

def retrieve_ke_candidates(
    query,
    search_k=1000
):

    query_vector = (
        product_query_model.encode(
            [query],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )
        .astype("float32")
    )

    scores, indices = (
        faiss_index.search(
            query_vector,
            search_k
        )
    )

    candidates = {}


    # ----------------------------
    # Base FAISS
    # ----------------------------

    for score, row_id in zip(
        scores[0],
        indices[0]
    ):

        row_id = int(row_id)

        if row_id < 0:
            continue

        meta = (
            metadata_by_faiss_row
            .loc[row_id]
        )

        product_id = str(
            meta["product_id"]
        )

        candidates[product_id] = {
            "product_id": product_id,
            "faiss_row_id": row_id,
            "semantic_score": float(score),
            "score_source": "base_faiss"
        }


    # ----------------------------
    # KE overlay
    # ----------------------------

    q = query_vector[0]

    for product_id, data in (
        ke_vector_overlay.items()
    ):

        edited_score = float(
            np.dot(
                q,
                data["vector"]
            )
        )

        if product_id in candidates:

            candidates[
                product_id
            ][
                "base_semantic_score"
            ] = candidates[
                product_id
            ][
                "semantic_score"
            ]

        else:

            candidates[
                product_id
            ] = {
                "product_id": product_id,
                "faiss_row_id": None
            }

        candidates[
            product_id
        ][
            "semantic_score"
        ] = edited_score

        candidates[
            product_id
        ][
            "score_source"
        ] = "ke_overlay"


    return list(
        candidates.values()
    )


print(
    "PASS: KE-AWARE CANDIDATE RETRIEVAL READY"
)

PASS: KE-AWARE CANDIDATE RETRIEVAL READY


In [54]:
# ============================================================
# CELL 48 — CONSTRAINT + SEMANTIC RANKING
# ============================================================

def ke_semantic_search(
    query,
    top_k=200,
    search_k=1000
):

    req = (
        parse_ke_query_requirements(
            query
        )
    )

    candidates = (
        retrieve_ke_candidates(
            query,
            search_k=search_k
        )
    )

    eligible = []


    for candidate in candidates:

        pid = candidate[
            "product_id"
        ]

        result = (
            get_effective_product_knowledge(
                pid
            )
        )

        p = result["effective"]


        # -------------------------
        # Capacity
        # -------------------------

        if (
            req["capacity_gb"]
            is not None
        ):

            capacities = (
                extract_capacities_gb(
                    p.get(
                        "product_name"
                    )
                )
            )

            if (
                req["capacity_gb"]
                not in capacities
            ):
                continue


        # -------------------------
        # Currency
        # -------------------------

        currency = normalize_value(
            p.get("currency")
        )

        if (
            req["currency"]
            is not None
            and
            str(currency).upper()
            != req["currency"]
        ):
            continue


        # -------------------------
        # Price
        # -------------------------

        price = normalize_value(
            p.get("price")
        )

        if (
            req["max_price"]
            is not None
        ):

            if price is None:
                continue

            try:
                price = float(price)
            except Exception:
                continue

            if (
                price
                > req["max_price"]
            ):
                continue


        # -------------------------
        # Rating
        # -------------------------

        rating = normalize_value(
            p.get(
                "average_rating"
            )
        )

        try:
            rating = float(rating)
        except Exception:
            rating = 0.0


        # -------------------------
        # Review count
        # -------------------------

        review_count = normalize_value(
            p.get("review_count")
        )

        try:
            review_count = max(
                0.0,
                float(review_count)
            )
        except Exception:
            review_count = 0.0


        semantic = float(
            candidate[
                "semantic_score"
            ]
        )


        review_norm = (
            np.log1p(
                review_count
            )
            /
            np.log1p(
                1_000_000
            )
        )


        final_score = (
            0.75 * semantic
            +
            0.20 * (
                rating / 5.0
            )
            +
            0.05 * review_norm
        )


        eligible.append({
            "product_id": pid,

            "title": normalize_value(
                p.get(
                    "product_name"
                )
            ),

            "source": normalize_value(
                p.get("source")
            ),

            "price": price,

            "currency": currency,

            "average_rating": rating,

            "review_count": int(
                review_count
            ),

            "semantic_score": semantic,

            "final_score": float(
                final_score
            ),

            "score_source": (
                candidate[
                    "score_source"
                ]
            ),

            "knowledge_edit_applied": (
                len(
                    result[
                        "applied_edits"
                    ]
                ) > 0
            )
        })


    eligible.sort(
        key=lambda x: x[
            "final_score"
        ],
        reverse=True
    )


    return {
        "query": query,
        "requirements": req,

        "searched_candidates": (
            len(candidates)
        ),

        "eligible_candidates": (
            len(eligible)
        ),

        "products": (
            eligible[:top_k]
        )
    }


print(
    "PASS: KE SEMANTIC RANKER READY"
)

PASS: KE SEMANTIC RANKER READY


In [55]:
# ============================================================
# CELL 49 — NO-EDIT REGRESSION TEST
# ============================================================

TEST_QUERY_FINAL = (
    "microSD 64GB under 20 USD "
    "for photos and video"
)


assert len(
    ke_vector_overlay
) == 0


regression_result = (
    ke_semantic_search(
        query=TEST_QUERY_FINAL,
        top_k=5,
        search_k=1000
    )
)


print("=" * 80)
print("NO-EDIT REGRESSION TEST")
print("=" * 80)

print(
    "QUERY:",
    regression_result["query"]
)

print(
    "\nREQUIREMENTS:"
)

print(
    regression_result[
        "requirements"
    ]
)

print(
    "\nSEARCHED:",
    regression_result[
        "searched_candidates"
    ]
)

print(
    "ELIGIBLE:",
    regression_result[
        "eligible_candidates"
    ]
)


print("\nTOP 5")

for i, p in enumerate(
    regression_result[
        "products"
    ],
    start=1
):

    print(
        f"\n#{i}"
    )

    print(
        p["product_id"]
    )

    print(
        p["title"]
    )

    print(
        p["price"],
        p["currency"]
    )

    print(
        "Rating:",
        p["average_rating"]
    )

    print(
        "Semantic:",
        round(
            p["semantic_score"],
            4
        )
    )

    print(
        "Final:",
        round(
            p["final_score"],
            4
        )
    )

    print(
        "KE:",
        p[
            "knowledge_edit_applied"
        ]
    )


assert all(
    not p[
        "knowledge_edit_applied"
    ]
    for p in regression_result[
        "products"
    ]
)


print(
    "\nPASS: NO ACTIVE EDIT "
    "CONTAMINATES BASELINE"
)

NO-EDIT REGRESSION TEST
QUERY: microSD 64GB under 20 USD for photos and video

REQUIREMENTS:
{'original_query': 'microSD 64GB under 20 USD for photos and video', 'capacity_gb': 64, 'max_price': 20.0, 'currency': 'USD'}

SEARCHED: 1000
ELIGIBLE: 80

TOP 5

#1
amazon_B0BRZPCQNJ
SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
16.99 USD
Rating: 4.7
Semantic: 0.7204
Final: 0.7707
KE: False

#2
amazon_B0C1H1Q3C1
Hugolog 64GB Micro SD Card, Micro SDXC UHS-I Memory Card for LaView Camera– 95MB/s,633X,U3,C10, Full HD Video V30, A1, FAT32, High Speed Flash TF Card P500 for Phone/Tablet/PC/Computer
7.99 USD
Rating: 4.6
Semantic: 0.717
Final: 0.7451
KE: False

#3
amazon_B07XDCZ9J3
SanDisk 64GB X2 (128GB) MicroSD HC Ultra Uhs-1 Memory Card
12.78 USD
Rating: 4.7
Semantic: 0.6965
Final: 0.7438
KE: False

#4
amazon_B09HKJY2QB
Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card 

In [56]:
# ============================================================
# CELL 50 — KE-AWARE BPR HYBRID RERANK
# ============================================================

def ke_hybrid_rerank(
    search_result,
    user_id=None
):

    items = [
        dict(p)
        for p in search_result["products"]
    ]

    # Lưu semantic ranking score
    for item in items:

        item["semantic_rank_score"] = (
            float(item["final_score"])
        )

        item["hybrid_score"] = (
            float(item["final_score"])
        )

        item["bpr_known"] = False
        item["bpr_score"] = None
        item["bpr_adjustment"] = 0.0


    # ---------------------------------
    # Không có user
    # ---------------------------------

    if user_id is None:

        return {
            "mode": "semantic",
            "user_id": None,
            "bpr_known_candidates": 0,
            "products": items
        }


    user_id = str(user_id)


    # ---------------------------------
    # Unknown user
    # ---------------------------------

    if user_id not in user_to_idx:

        return {
            "mode": "semantic_unknown_user",
            "user_id": user_id,
            "bpr_known_candidates": 0,
            "products": items
        }


    user_idx = int(
        user_to_idx[user_id]
    )

    user_vector = (
        bpr_model.user_factors[
            user_idx
        ]
    )


    known_positions = []
    raw_scores = []


    # ---------------------------------
    # Compute BPR scores
    # ---------------------------------

    for pos, item in enumerate(items):

        pid = str(
            item["product_id"]
        )

        if pid not in item_to_idx:
            continue

        item_idx = int(
            item_to_idx[pid]
        )

        item_vector = (
            bpr_model.item_factors[
                item_idx
            ]
        )

        score = float(
            np.dot(
                user_vector,
                item_vector
            )
        )

        item["bpr_known"] = True
        item["bpr_score"] = score

        known_positions.append(pos)
        raw_scores.append(score)


    # Step 12 rule:
    # cần >= 2 BPR-known candidates
    if len(raw_scores) < 2:

        return {
            "mode": "semantic_bpr_insufficient",
            "user_id": user_id,
            "bpr_known_candidates": len(
                raw_scores
            ),
            "products": items
        }


    # ---------------------------------
    # Percentile rank
    # ---------------------------------

    score_series = pd.Series(
        raw_scores,
        dtype=float
    )

    percentiles = (
        score_series
        .rank(
            method="average",
            pct=True
        )
        .to_numpy()
    )

    mean_percentile = float(
        np.mean(percentiles)
    )


    # ---------------------------------
    # Bounded adjustment
    # ---------------------------------

    for pos, percentile in zip(
        known_positions,
        percentiles
    ):

        adjustment = (
            0.04
            * (
                float(percentile)
                - mean_percentile
            )
        )

        items[pos][
            "bpr_adjustment"
        ] = float(adjustment)

        items[pos][
            "hybrid_score"
        ] = (
            items[pos][
                "semantic_rank_score"
            ]
            + adjustment
        )


    items.sort(
        key=lambda x: x[
            "hybrid_score"
        ],
        reverse=True
    )


    return {
        "mode": "hybrid_bpr",
        "user_id": user_id,
        "bpr_known_candidates": len(
            raw_scores
        ),
        "products": items
    }


print(
    "PASS: KE-AWARE BPR RERANK READY"
)

PASS: KE-AWARE BPR RERANK READY


In [57]:
# ============================================================
# CELL 51 — FULL KE + HYBRID SEARCH
# ============================================================

def ke_hybrid_search(
    query,
    user_id=None,
    top_k=5,
    candidate_k=200,
    search_k=1000
):

    # Refresh active KE vectors
    global ke_vector_overlay

    ke_vector_overlay = (
        build_ke_vector_overlay()
    )


    # ---------------------------------
    # Semantic candidate pool
    # ---------------------------------

    semantic_result = (
        ke_semantic_search(
            query=query,
            top_k=candidate_k,
            search_k=search_k
        )
    )


    # ---------------------------------
    # BPR rerank
    # ---------------------------------

    hybrid = (
        ke_hybrid_rerank(
            semantic_result,
            user_id=user_id
        )
    )


    final_products = (
        hybrid["products"][
            :top_k
        ]
    )


    return {
        "query": query,

        "requirements": (
            semantic_result[
                "requirements"
            ]
        ),

        "mode": hybrid["mode"],

        "user_id": user_id,

        "search_k": search_k,

        "candidate_k": candidate_k,

        "top_k": top_k,

        "searched_candidates": (
            semantic_result[
                "searched_candidates"
            ]
        ),

        "eligible_candidates": (
            semantic_result[
                "eligible_candidates"
            ]
        ),

        "bpr_known_candidates": (
            hybrid[
                "bpr_known_candidates"
            ]
        ),

        "active_ke_products": len(
            ke_vector_overlay
        ),

        "products": final_products
    }


print(
    "PASS: FULL KE + HYBRID SEARCH READY"
)

PASS: FULL KE + HYBRID SEARCH READY


In [58]:
# ============================================================
# CELL 52 — NEW USER TEST
# ============================================================

new_user_result = (
    ke_hybrid_search(
        query=TEST_QUERY_FINAL,
        user_id=None,
        top_k=5,
        candidate_k=200,
        search_k=1000
    )
)


print("=" * 80)
print("NEW USER / SEMANTIC MODE")
print("=" * 80)

print(
    "MODE:",
    new_user_result["mode"]
)

print(
    "ELIGIBLE:",
    new_user_result[
        "eligible_candidates"
    ]
)

print(
    "ACTIVE KE PRODUCTS:",
    new_user_result[
        "active_ke_products"
    ]
)


for i, p in enumerate(
    new_user_result["products"],
    start=1
):

    print(
        f"\n#{i}",
        p["product_id"]
    )

    print(
        p["title"]
    )

    print(
        "Semantic rank:",
        round(
            p["semantic_rank_score"],
            4
        )
    )

    print(
        "Hybrid:",
        round(
            p["hybrid_score"],
            4
        )
    )

    print(
        "BPR known:",
        p["bpr_known"]
    )

    print(
        "KE:",
        p[
            "knowledge_edit_applied"
        ]
    )


assert (
    new_user_result["mode"]
    == "semantic"
)

assert (
    new_user_result[
        "active_ke_products"
    ] == 0
)


print(
    "\nPASS: NEW USER FALLBACK WORKS"
)

NEW USER / SEMANTIC MODE
MODE: semantic
ELIGIBLE: 80
ACTIVE KE PRODUCTS: 0

#1 amazon_B0BRZPCQNJ
SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
Semantic rank: 0.7707
Hybrid: 0.7707
BPR known: False
KE: False

#2 amazon_B0C1H1Q3C1
Hugolog 64GB Micro SD Card, Micro SDXC UHS-I Memory Card for LaView Camera– 95MB/s,633X,U3,C10, Full HD Video V30, A1, FAT32, High Speed Flash TF Card P500 for Phone/Tablet/PC/Computer
Semantic rank: 0.7451
Hybrid: 0.7451
BPR known: False
KE: False

#3 amazon_B07XDCZ9J3
SanDisk 64GB X2 (128GB) MicroSD HC Ultra Uhs-1 Memory Card
Semantic rank: 0.7438
Hybrid: 0.7438
BPR known: False
KE: False

#4 amazon_B09HKJY2QB
Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
Semantic rank: 0.7393
Hybrid: 0.7393
BPR known: False
KE: False

#5 amazon_B01LORO7BA
SanDisk 64GB Extreme SDXC UHS-I Memory Card - 90MB/s, C10, U3, V30, 4

In [59]:
# ============================================================
# CELL 53 — KNOWN USER HYBRID TEST
# ============================================================

TEST_KNOWN_USER = (
    "amazon_AE22DOLTN2WN6B3TJ32UDXOEFNSA"
)


assert TEST_KNOWN_USER in (
    user_to_idx
)


known_user_result = (
    ke_hybrid_search(
        query=TEST_QUERY_FINAL,
        user_id=TEST_KNOWN_USER,
        top_k=5,
        candidate_k=200,
        search_k=1000
    )
)


print("=" * 80)
print("KNOWN USER / HYBRID BPR")
print("=" * 80)

print(
    "USER:",
    TEST_KNOWN_USER
)

print(
    "MODE:",
    known_user_result["mode"]
)

print(
    "ELIGIBLE:",
    known_user_result[
        "eligible_candidates"
    ]
)

print(
    "BPR-KNOWN CANDIDATES:",
    known_user_result[
        "bpr_known_candidates"
    ]
)

print(
    "ACTIVE KE PRODUCTS:",
    known_user_result[
        "active_ke_products"
    ]
)


for i, p in enumerate(
    known_user_result["products"],
    start=1
):

    print(
        f"\n#{i}"
    )

    print(
        p["product_id"]
    )

    print(
        p["title"]
    )

    print(
        p["price"],
        p["currency"]
    )

    print(
        "Semantic rank:",
        round(
            p[
                "semantic_rank_score"
            ],
            4
        )
    )

    print(
        "BPR known:",
        p["bpr_known"]
    )

    print(
        "BPR score:",
        (
            round(
                p["bpr_score"],
                6
            )
            if p["bpr_score"]
            is not None
            else None
        )
    )

    print(
        "Adjustment:",
        round(
            p[
                "bpr_adjustment"
            ],
            6
        )
    )

    print(
        "Hybrid:",
        round(
            p["hybrid_score"],
            4
        )
    )

    print(
        "KE:",
        p[
            "knowledge_edit_applied"
        ]
    )


assert (
    known_user_result["mode"]
    == "hybrid_bpr"
)

assert (
    known_user_result[
        "bpr_known_candidates"
    ] >= 2
)


print(
    "\nPASS: BPR PERSONALIZATION ACTIVE"
)

KNOWN USER / HYBRID BPR
USER: amazon_AE22DOLTN2WN6B3TJ32UDXOEFNSA
MODE: hybrid_bpr
ELIGIBLE: 80
BPR-KNOWN CANDIDATES: 26
ACTIVE KE PRODUCTS: 0

#1
amazon_B0BRZPCQNJ
SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
16.99 USD
Semantic rank: 0.7707
BPR known: True
BPR score: -0.455709
Adjustment: -0.013077
Hybrid: 0.7576
KE: False

#2
amazon_B09HKJY2QB
Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
6.49 USD
Semantic rank: 0.7393
BPR known: True
BPR score: 0.057965
Adjustment: 0.006923
Hybrid: 0.7462
KE: False

#3
amazon_B0C1H1Q3C1
Hugolog 64GB Micro SD Card, Micro SDXC UHS-I Memory Card for LaView Camera– 95MB/s,633X,U3,C10, Full HD Video V30, A1, FAT32, High Speed Flash TF Card P500 for Phone/Tablet/PC/Computer
7.99 USD
Semantic rank: 0.7451
BPR known: False
BPR score: None
Adjustment: 0.0
Hybrid: 0.7451
KE: False

#4
amazon_B07XDCZ9J3
SanD

# Chuẩn bị Review RAG

In [60]:
# ============================================================
# CELL 54 — PREPARE REVIEW RAG
# ============================================================

reviews_ke = reviews.copy()

reviews_ke["rag_review_text"] = (
    reviews_ke["smart_review_text"]
    .fillna(
        reviews_ke["review_text"]
    )
    .fillna("")
    .astype(str)
    .str.strip()
)

reviews_ke = reviews_ke[
    reviews_ke["rag_review_text"] != ""
].copy()


reviews_by_product_ke = {
    str(pid): group.copy()
    for pid, group
    in reviews_ke.groupby(
        "product_id",
        sort=False
    )
}


print("=" * 80)
print("REVIEW RAG PREPARATION")
print("=" * 80)

print(
    "TOTAL REVIEWS:",
    len(reviews)
)

print(
    "USABLE REVIEW TEXT:",
    len(reviews_ke)
)

print(
    "PRODUCTS WITH REVIEWS:",
    len(reviews_by_product_ke)
)

print(
    "\nTEST PRODUCT REVIEWS:",
    len(
        reviews_by_product_ke.get(
            TEST_PRODUCT_ID,
            []
        )
    )
)

print(
    "\nPASS: REVIEW RAG DATA READY"
)

REVIEW RAG PREPARATION
TOTAL REVIEWS: 767179
USABLE REVIEW TEXT: 766776
PRODUCTS WITH REVIEWS: 123310

TEST PRODUCT REVIEWS: 33

PASS: REVIEW RAG DATA READY


In [61]:
# ============================================================
# CELL 55 — REVIEW RETRIEVAL
# ============================================================

def retrieve_ke_reviews(
    product_id,
    query,
    max_reviews=3,
    candidate_limit=300
):

    product_id = str(product_id)

    group = reviews_by_product_ke.get(
        product_id
    )

    if group is None or len(group) == 0:
        return []


    candidates = (
        group.iloc[
            :candidate_limit
        ].copy()
    )

    texts = (
        candidates[
            "rag_review_text"
        ].tolist()
    )

    if not texts:
        return []


    query_vec = (
        product_query_model.encode(
            [query],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )[0]
    )


    review_vecs = (
        product_query_model.encode(
            texts,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
            batch_size=64
        )
    )


    similarities = (
        review_vecs
        @ query_vec
    )


    candidates[
        "_relevance"
    ] = similarities


    candidates = (
        candidates
        .sort_values(
            "_relevance",
            ascending=False
        )
        .head(max_reviews)
    )


    results = []

    for _, row in (
        candidates.iterrows()
    ):

        rating = normalize_value(
            row.get(
                "smart_review_rating",
                row.get("rating")
            )
        )

        results.append({
            "review_id": str(
                row["review_id"]
            ),

            "review_rating": rating,

            "review_text": (
                str(
                    row[
                        "rag_review_text"
                    ]
                )[:500]
            ),

            "relevance_score": float(
                row["_relevance"]
            )
        })


    return results


print(
    "PASS: REVIEW RETRIEVAL READY"
)

PASS: REVIEW RETRIEVAL READY


In [62]:
# ============================================================
# CELL 56 — BUILD KE + REVIEW RAG CONTEXT
# ============================================================

def build_ke_rag_context(
    search_result,
    max_reviews=3
):

    contexts = []


    for item in (
        search_result["products"]
    ):

        pid = str(
            item["product_id"]
        )


        # Effective knowledge
        knowledge = (
            get_effective_product_knowledge(
                pid
            )
        )

        p = knowledge[
            "effective"
        ]


        # Review RAG
        retrieved_reviews = (
            retrieve_ke_reviews(
                product_id=pid,
                query=search_result[
                    "query"
                ],
                max_reviews=max_reviews
            )
        )


        linked_review_count = len(
            reviews_by_product_ke.get(
                pid,
                []
            )
        )


        context = {
            "product_id": pid,

            "title": normalize_value(
                p.get("product_name")
            ),

            "source": normalize_value(
                p.get("source")
            ),

            "price": normalize_value(
                p.get("price")
            ),

            "currency": normalize_value(
                p.get("currency")
            ),

            "product_average_rating": (
                normalize_value(
                    p.get(
                        "average_rating"
                    )
                )
            ),

            "metadata_rating_count": (
                normalize_value(
                    p.get(
                        "review_count"
                    )
                )
            ),

            "linked_review_count": (
                linked_review_count
            ),

            "retrieved_evidence_count": (
                len(retrieved_reviews)
            ),

            "product_url": normalize_value(
                p.get("product_url")
            ),

            "description": (
                clean_text_value(
                    p.get("description")
                )
            ),

            "features": (
                clean_text_value(
                    p.get(
                        "product_features"
                    )
                )
            ),

            "knowledge_edit": {
                "applied": (
                    len(
                        knowledge[
                            "applied_edits"
                        ]
                    ) > 0
                ),

                "active_edit_count": (
                    knowledge[
                        "active_edit_count"
                    ]
                ),

                "fields": (
                    knowledge[
                        "applied_edits"
                    ]
                )
            },

            "ranking": {
                "semantic_rank_score": (
                    item.get(
                        "semantic_rank_score"
                    )
                ),

                "bpr_known": (
                    item.get(
                        "bpr_known"
                    )
                ),

                "bpr_adjustment": (
                    item.get(
                        "bpr_adjustment"
                    )
                ),

                "hybrid_score": (
                    item.get(
                        "hybrid_score"
                    )
                )
            },

            "reviews": (
                retrieved_reviews
            )
        }


        contexts.append(
            context
        )


    return {
        "query": search_result["query"],
        "mode": search_result["mode"],
        "user_id": search_result["user_id"],
        "products": contexts
    }


print(
    "PASS: KE + REVIEW RAG BUILDER READY"
)

PASS: KE + REVIEW RAG BUILDER READY


In [63]:
# ============================================================
# CELL 57 — PRODUCT / REVIEW RELATION AUDIT
# ============================================================

def audit_ke_review_relations(
    rag_context
):

    checked = 0
    errors = []


    for product in (
        rag_context["products"]
    ):

        pid = str(
            product["product_id"]
        )

        valid_ids = set()

        group = (
            reviews_by_product_ke.get(
                pid
            )
        )

        if group is not None:

            valid_ids = set(
                group[
                    "review_id"
                ].astype(str)
            )


        for review in (
            product["reviews"]
        ):

            checked += 1

            rid = str(
                review["review_id"]
            )

            if rid not in valid_ids:

                errors.append({
                    "product_id": pid,
                    "review_id": rid
                })


    return {
        "checked_reviews": checked,
        "errors": errors,
        "passed": len(errors) == 0
    }


print(
    "PASS: REVIEW RELATION AUDIT READY"
)

PASS: REVIEW RELATION AUDIT READY


# END-TO-END RAG CONTEXT TEST


In [64]:
# ============================================================
# CELL 58 — END-TO-END RAG CONTEXT TEST
# ============================================================

final_rag_context = (
    build_ke_rag_context(
        known_user_result,
        max_reviews=3
    )
)


review_relation_audit = (
    audit_ke_review_relations(
        final_rag_context
    )
)


print("=" * 80)
print("KE + HYBRID + REVIEW RAG")
print("=" * 80)

print(
    "MODE:",
    final_rag_context["mode"]
)

print(
    "PRODUCT COUNT:",
    len(
        final_rag_context[
            "products"
        ]
    )
)

print(
    "REVIEW RELATION PASS:",
    review_relation_audit[
        "passed"
    ]
)

print(
    "CHECKED REVIEWS:",
    review_relation_audit[
        "checked_reviews"
    ]
)


for i, p in enumerate(
    final_rag_context[
        "products"
    ],
    start=1
):

    print("\n" + "-" * 80)

    print(
        f"#{i}",
        p["product_id"]
    )

    print(
        p["title"]
    )

    print(
        "PRICE:",
        p["price"],
        p["currency"]
    )

    print(
        "RATING:",
        p[
            "product_average_rating"
        ]
    )

    print(
        "LINKED REVIEWS:",
        p[
            "linked_review_count"
        ]
    )

    print(
        "RETRIEVED:",
        p[
            "retrieved_evidence_count"
        ]
    )

    print(
        "KE APPLIED:",
        p[
            "knowledge_edit"
        ]["applied"]
    )

    print(
        "BPR KNOWN:",
        p[
            "ranking"
        ]["bpr_known"]
    )

    print("REVIEW IDS:")

    for r in p["reviews"]:

        print(
            " -",
            r["review_id"],
            "| relevance:",
            round(
                r[
                    "relevance_score"
                ],
                4
            )
        )


assert (
    review_relation_audit[
        "passed"
    ]
)

assert (
    len(
        final_rag_context[
            "products"
        ]
    ) == 5
)

assert all(
    not p[
        "knowledge_edit"
    ]["applied"]
    for p in final_rag_context[
        "products"
    ]
)


print(
    "\nPASS: KE + HYBRID + "
    "REVIEW RAG PIPELINE READY"
)

KE + HYBRID + REVIEW RAG
MODE: hybrid_bpr
PRODUCT COUNT: 5
REVIEW RELATION PASS: True
CHECKED REVIEWS: 14

--------------------------------------------------------------------------------
#1 amazon_B0BRZPCQNJ
SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
PRICE: 16.99 USD
RATING: 4.7
LINKED REVIEWS: 46
RETRIEVED: 3
KE APPLIED: False
BPR KNOWN: True
REVIEW IDS:
 - review_0126056 | relevance: 0.676
 - review_0077221 | relevance: 0.5396
 - review_0008587 | relevance: 0.487

--------------------------------------------------------------------------------
#2 amazon_B09HKJY2QB
Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
PRICE: 6.49 USD
RATING: 4.6
LINKED REVIEWS: 33
RETRIEVED: 3
KE APPLIED: False
BPR KNOWN: True
REVIEW IDS:
 - review_0482507 | relevance: 0.6579
 - review_0374815 | relevance: 0.6121
 - review_0360039 | relevance: 0.5416

--

#  LOAD QWEN


In [65]:
# ============================================================
# CELL 59A — CHECK LLM ENVIRONMENT
# ============================================================

import sys
import torch
import transformers

print("=" * 80)
print("LLM ENVIRONMENT")
print("=" * 80)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

try:
    import bitsandbytes as bnb
    print("bitsandbytes:", bnb.__version__)
except Exception as e:
    print("bitsandbytes: NOT AVAILABLE")
    print("Error:", repr(e))

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

LLM ENVIRONMENT
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
Transformers: 5.0.0
bitsandbytes: 0.50.2

CUDA available: True
GPU: Tesla T4


In [66]:
# ============================================================
# CELL 59B — INSTALL BITSANDBYTES
# ============================================================

!pip install -q "bitsandbytes>=0.46.1"

print(
    "PASS: bitsandbytes installation command completed"
)

PASS: bitsandbytes installation command completed


In [67]:
# ============================================================
# CELL 59C — VERIFY LLM ENVIRONMENT
# ============================================================

import torch
import transformers
import bitsandbytes as bnb

print("=" * 80)
print("LLM ENVIRONMENT AFTER RESTART")
print("=" * 80)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "Transformers:",
    transformers.__version__
)

print(
    "bitsandbytes:",
    bnb.__version__
)

print(
    "CUDA:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


from transformers import (
    BitsAndBytesConfig,
    AutoTokenizer,
    AutoModelForCausalLM
)


print(
    "\nPASS: LLM DEPENDENCIES IMPORTED"
)

LLM ENVIRONMENT AFTER RESTART
PyTorch: 2.10.0+cu128
Transformers: 5.0.0
bitsandbytes: 0.50.2
CUDA: True
GPU: Tesla T4

PASS: LLM DEPENDENCIES IMPORTED


In [68]:
# ============================================================
# CELL 59 — LOAD QWEN 2.5 3B (UPDATED ENVIRONMENT)
# ============================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)


LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"


print("=" * 80)
print("LOADING QWEN")
print("=" * 80)

print("MODEL:", LLM_MODEL_NAME)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ------------------------------------------------------------
# 4-bit quantization
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)


# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME
)


# ------------------------------------------------------------
# Model
# Transformers 5.x: use dtype instead of torch_dtype
# ------------------------------------------------------------

print("Loading model in 4-bit...")

llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,

    quantization_config=bnb_config,

    device_map="auto",

    dtype=torch.float16
)


llm.eval()


print("\n" + "=" * 80)
print("QWEN LOADED")
print("=" * 80)

print(
    "MODEL CLASS:",
    type(llm).__name__
)

print(
    "TOKENIZER:",
    type(tokenizer).__name__
)

print(
    "MODEL DEVICE:",
    next(llm.parameters()).device
)

print(
    "DTYPE:",
    next(llm.parameters()).dtype
)


# ------------------------------------------------------------
# Small generation smoke test
# ------------------------------------------------------------

messages = [
    {
        "role": "user",
        "content": "Trả lời đúng một từ: OK"
    }
]


smoke_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)


smoke_inputs = tokenizer(
    smoke_prompt,
    return_tensors="pt"
).to(
    next(llm.parameters()).device
)


with torch.no_grad():

    smoke_output = llm.generate(
        **smoke_inputs,
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )


new_tokens = smoke_output[
    0,
    smoke_inputs["input_ids"].shape[1]:
]


smoke_text = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()


print(
    "\nSMOKE TEST OUTPUT:",
    repr(smoke_text)
)

print(
    "\nPASS: QWEN READY"
)

LOADING QWEN
MODEL: Qwen/Qwen2.5-3B-Instruct
CUDA: True
GPU: Tesla T4

Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



QWEN LOADED
MODEL CLASS: Qwen2ForCausalLM
TOKENIZER: Qwen2Tokenizer
MODEL DEVICE: cuda:0
DTYPE: torch.float16

SMOKE TEST OUTPUT: 'OK'

PASS: QWEN READY


In [69]:
# ============================================================
# CELL 60 — DETERMINISTIC PRODUCT CARDS
# ============================================================

def format_ke_product_cards(
    rag_context
):

    blocks = []

    for i, p in enumerate(
        rag_context["products"],
        start=1
    ):

        lines = [
            f"{i}. {p['title']}",
            (
                f"   Product ID: "
                f"{p['product_id']}"
            ),
            (
                f"   Giá dữ liệu: "
                f"{p['price']} "
                f"{p['currency']}"
            ),
            (
                f"   Rating: "
                f"{p['product_average_rating']}/5"
            ),
            (
                f"   Số rating metadata: "
                f"{p['metadata_rating_count']}"
            ),
            (
                f"   Review liên kết: "
                f"{p['linked_review_count']}"
            )
        ]

        if p[
            "knowledge_edit"
        ]["applied"]:

            fields = list(
                p[
                    "knowledge_edit"
                ]["fields"].keys()
            )

            lines.append(
                "   Knowledge Edit: "
                + ", ".join(fields)
            )

        blocks.append(
            "\n".join(lines)
        )

    return "\n\n".join(
        blocks
    )


product_cards = (
    format_ke_product_cards(
        final_rag_context
    )
)


print(product_cards)

1. SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
   Product ID: amazon_B0BRZPCQNJ
   Giá dữ liệu: 16.99 USD
   Rating: 4.7/5
   Số rating metadata: 121509
   Review liên kết: 46

2. Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
   Product ID: amazon_B09HKJY2QB
   Giá dữ liệu: 6.49 USD
   Rating: 4.6/5
   Số rating metadata: 15428
   Review liên kết: 33

3. Hugolog 64GB Micro SD Card, Micro SDXC UHS-I Memory Card for LaView Camera– 95MB/s,633X,U3,C10, Full HD Video V30, A1, FAT32, High Speed Flash TF Card P500 for Phone/Tablet/PC/Computer
   Product ID: amazon_B0C1H1Q3C1
   Giá dữ liệu: 7.99 USD
   Rating: 4.6/5
   Số rating metadata: 636
   Review liên kết: 2

4. SanDisk 64GB X2 (128GB) MicroSD HC Ultra Uhs-1 Memory Card
   Product ID: amazon_B07XDCZ9J3
   Giá dữ liệu: 12.78 USD
   Rating: 4.7/5
   Số rating metadata: 10080
   Review 

In [70]:
# ============================================================
# CELL 61 — BUILD GROUNDED LLM PROMPT
# ============================================================

def build_ke_llm_prompt(
    rag_context
):

    evidence = []


    for p in (
        rag_context["products"][:3]
    ):

        product_block = {
            "product_id": (
                p["product_id"]
            ),

            "title": p["title"],

            "description": (
                p["description"][:500]
                if p["description"]
                else ""
            ),

            "features": (
                p["features"][:800]
                if p["features"]
                else ""
            ),

            "knowledge_edit": (
                p["knowledge_edit"]
            ),

            "reviews": (
                p["reviews"][:2]
            )
        }

        evidence.append(
            product_block
        )


    system_prompt = """
Bạn là lớp diễn giải của hệ thống gợi ý sản phẩm.

QUY TẮC BẮT BUỘC:

1. Chỉ sử dụng thông tin có trong CONTEXT.
2. Không tự tạo giá, rating, tốc độ, tính năng hoặc khả năng tương thích.
3. Không nói "chắc chắn", "hoàn toàn", "yên tâm" hoặc đưa ra bảo đảm.
4. Nếu sử dụng nội dung review, phải ghi chính xác [review_id].
5. Review chỉ là trải nghiệm của người dùng, không phải sự bảo đảm chung.
6. Không tự suy ra thông tin không có trong context.
7. Không viết lại giá hoặc rating; phần đó do hệ thống deterministic hiển thị.
8. Chỉ viết tối đa 3 câu ngắn bằng tiếng Việt.
9. Nếu evidence không đủ, hãy nói evidence hiện có còn hạn chế.
""".strip()


    user_prompt = f"""
QUERY:
{rag_context['query']}

CONTEXT:
{json.dumps(
    evidence,
    ensure_ascii=False,
    default=str
)}

Hãy đưa ra nhận xét ngắn để hỗ trợ người dùng so sánh các sản phẩm.
""".strip()


    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]


print(
    "PASS: GROUNDED PROMPT BUILDER READY"
)

PASS: GROUNDED PROMPT BUILDER READY


In [71]:
# ============================================================
# CELL 62 — GENERATE QWEN DRAFT
# ============================================================

def generate_ke_commentary(
    rag_context,
    max_new_tokens=180
):

    messages = (
        build_ke_llm_prompt(
            rag_context
        )
    )


    prompt = (
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    )


    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(
        next(
            llm.parameters()
        ).device
    )


    with torch.no_grad():

        outputs = llm.generate(
            **inputs,

            max_new_tokens=(
                max_new_tokens
            ),

            do_sample=False,

            pad_token_id=(
                tokenizer.eos_token_id
            )
        )


    generated = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]


    answer = (
        tokenizer.decode(
            generated,
            skip_special_tokens=True
        )
        .strip()
    )


    return {
        "text": answer,

        "prompt_tokens": int(
            inputs[
                "input_ids"
            ].shape[1]
        ),

        "generated_tokens": int(
            generated.shape[0]
        )
    }


ke_llm_draft = (
    generate_ke_commentary(
        final_rag_context
    )
)


print("=" * 80)
print("QWEN DRAFT")
print("=" * 80)

print(
    ke_llm_draft["text"]
)

print(
    "\nPROMPT TOKENS:",
    ke_llm_draft[
        "prompt_tokens"
    ]
)

print(
    "GENERATED TOKENS:",
    ke_llm_draft[
        "generated_tokens"
    ]
)

QWEN DRAFT
SanDisk và Micro Center Premium đều phù hợp, nhưng SanDisk có giá tốt hơn. Hugolog 64GB có tốc độ đọc và ghi cao, nhưng đánh giá khách hàng còn hạn chế.

PROMPT TOKENS: 2075
GENERATED TOKENS: 45


In [72]:
# ============================================================
# CELL 63 — RULE-BASED GROUNDING AUDIT
# ============================================================

import re


def audit_ke_commentary(
    commentary,
    rag_context
):

    text = str(commentary).strip()

    issues = []


    # --------------------------------------------------------
    # 1. Valid review IDs from retrieved evidence only
    # --------------------------------------------------------

    valid_review_ids = set()

    for product in rag_context["products"]:

        for review in product["reviews"]:

            valid_review_ids.add(
                str(review["review_id"])
            )


    cited_review_ids = set(
        re.findall(
            r"\[(review_\d+)\]",
            text
        )
    )


    invalid_review_ids = (
        cited_review_ids
        - valid_review_ids
    )


    if invalid_review_ids:

        issues.append(
            "invalid_review_citation"
        )


    # --------------------------------------------------------
    # 2. Detect review/evidence-style claims without citation
    # --------------------------------------------------------

    review_claim_terms = [
        "khách hàng",
        "người dùng",
        "review",
        "đánh giá khách hàng",
        "phản hồi",
        "trải nghiệm",
        "người mua"
    ]


    has_review_claim = any(
        term in text.lower()
        for term in review_claim_terms
    )


    if (
        has_review_claim
        and len(cited_review_ids) == 0
    ):

        issues.append(
            "review_claim_without_citation"
        )


    # --------------------------------------------------------
    # 3. Overconfident wording
    # --------------------------------------------------------

    overconfident_terms = [
        "chắc chắn",
        "hoàn toàn",
        "đảm bảo",
        "bảo đảm",
        "yên tâm",
        "tốt nhất",
        "hoàn hảo"
    ]


    detected_overconfidence = [
        term
        for term in overconfident_terms
        if term in text.lower()
    ]


    if detected_overconfidence:

        issues.append(
            "overconfident_language"
        )


    # --------------------------------------------------------
    # 4. LLM should not independently compare/interpret price
    # --------------------------------------------------------

    price_claim_patterns = [
        r"giá tốt hơn",
        r"rẻ hơn",
        r"đắt hơn",
        r"giá rẻ",
        r"giá cao hơn",
        r"giá thấp hơn",
        r"giá hợp lý",
        r"đáng tiền"
    ]


    detected_price_claims = []

    for pattern in price_claim_patterns:

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        ):

            detected_price_claims.append(
                pattern
            )


    if detected_price_claims:

        issues.append(
            "llm_price_interpretation"
        )


    # --------------------------------------------------------
    # 5. Potential unsupported qualitative performance claims
    # --------------------------------------------------------

    performance_patterns = [
        r"tốc độ .* cao",
        r"tốc độ .* nhanh",
        r"hiệu năng .* cao",
        r"hiệu suất .* cao",
        r"nhanh hơn",
        r"ổn định hơn",
        r"bền hơn"
    ]


    detected_performance_claims = []

    for pattern in performance_patterns:

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        ):

            detected_performance_claims.append(
                pattern
            )


    if detected_performance_claims:

        issues.append(
            "unsupported_qualitative_claim"
        )


    # --------------------------------------------------------
    # Result
    # --------------------------------------------------------

    issues = list(
        dict.fromkeys(issues)
    )


    return {

        "passed": (
            len(issues) == 0
        ),

        "issues": issues,

        "valid_review_ids": sorted(
            valid_review_ids
        ),

        "cited_review_ids": sorted(
            cited_review_ids
        ),

        "invalid_review_ids": sorted(
            invalid_review_ids
        ),

        "detected_overconfidence": (
            detected_overconfidence
        ),

        "detected_price_claims": (
            detected_price_claims
        ),

        "detected_performance_claims": (
            detected_performance_claims
        )
    }


commentary_audit = (
    audit_ke_commentary(
        ke_llm_draft["text"],
        final_rag_context
    )
)


print("=" * 80)
print("QWEN GROUNDING AUDIT")
print("=" * 80)

print(
    "PASSED:",
    commentary_audit["passed"]
)

print(
    "ISSUES:",
    commentary_audit["issues"]
)

print(
    "CITED REVIEW IDS:",
    commentary_audit[
        "cited_review_ids"
    ]
)

print(
    "INVALID REVIEW IDS:",
    commentary_audit[
        "invalid_review_ids"
    ]
)

print(
    "PRICE CLAIMS:",
    commentary_audit[
        "detected_price_claims"
    ]
)

print(
    "PERFORMANCE CLAIMS:",
    commentary_audit[
        "detected_performance_claims"
    ]
)

QWEN GROUNDING AUDIT
PASSED: False
ISSUES: ['review_claim_without_citation', 'llm_price_interpretation', 'unsupported_qualitative_claim']
CITED REVIEW IDS: []
INVALID REVIEW IDS: []
PRICE CLAIMS: ['giá tốt hơn']
PERFORMANCE CLAIMS: ['tốc độ .* cao']


In [73]:
# ============================================================
# CELL 64 — DETERMINISTIC SAFE FALLBACK
# ============================================================

def build_ke_safe_fallback(
    rag_context,
    max_products=3
):

    lines = [
        (
            "Nhận xét tự động từ LLM không vượt qua "
            "kiểm tra grounding, vì vậy hệ thống chỉ "
            "hiển thị evidence đã truy xuất:"
        )
    ]


    for i, product in enumerate(
        rag_context["products"][
            :max_products
        ],
        start=1
    ):

        review_ids = [
            review["review_id"]
            for review in product[
                "reviews"
            ]
        ]


        if review_ids:

            evidence_text = (
                ", ".join(
                    f"[{rid}]"
                    for rid in review_ids
                )
            )

        else:

            evidence_text = (
                "không có review evidence"
            )


        lines.append(
            (
                f"{i}. "
                f"{product['title']} — "
                f"{product['linked_review_count']} "
                f"review liên kết; "
                f"{product['retrieved_evidence_count']} "
                f"review được truy xuất: "
                f"{evidence_text}."
            )
        )


    lines.append(
        (
            "Các review trên là evidence được truy xuất "
            "theo độ liên quan với truy vấn; hệ thống "
            "không coi chúng là bảo đảm về chất lượng "
            "chung của sản phẩm."
        )
    )


    return "\n".join(lines)


safe_fallback = (
    build_ke_safe_fallback(
        final_rag_context
    )
)


print("=" * 80)
print("SAFE FALLBACK")
print("=" * 80)

print(safe_fallback)

SAFE FALLBACK
Nhận xét tự động từ LLM không vượt qua kiểm tra grounding, vì vậy hệ thống chỉ hiển thị evidence đã truy xuất:
1. SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT — 46 review liên kết; 3 review được truy xuất: [review_0126056], [review_0077221], [review_0008587].
2. Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB) — 33 review liên kết; 3 review được truy xuất: [review_0482507], [review_0374815], [review_0360039].
3. Hugolog 64GB Micro SD Card, Micro SDXC UHS-I Memory Card for LaView Camera– 95MB/s,633X,U3,C10, Full HD Video V30, A1, FAT32, High Speed Flash TF Card P500 for Phone/Tablet/PC/Computer — 2 review liên kết; 2 review được truy xuất: [review_0016087], [review_0565323].
Các review trên là evidence được truy xuất theo độ liên quan với truy vấn; hệ thống không coi chúng là bảo đảm về chất lượng chung của sản phẩm.


In [74]:
# ============================================================
# CELL 65 — SELECT GROUNDED COMMENTARY
# ============================================================

def select_ke_commentary(
    llm_draft,
    rag_context
):

    audit = (
        audit_ke_commentary(
            llm_draft,
            rag_context
        )
    )


    if audit["passed"]:

        return {
            "source": "llm",
            "text": llm_draft,
            "audit": audit
        }


    fallback = (
        build_ke_safe_fallback(
            rag_context
        )
    )


    return {
        "source": (
            "deterministic_fallback"
        ),

        "text": fallback,

        "audit": audit
    }


selected_commentary = (
    select_ke_commentary(
        ke_llm_draft["text"],
        final_rag_context
    )
)


print("=" * 80)
print("COMMENTARY SELECTION")
print("=" * 80)

print(
    "SOURCE:",
    selected_commentary["source"]
)

print(
    "DRAFT AUDIT PASSED:",
    selected_commentary[
        "audit"
    ]["passed"]
)

print(
    "ISSUES:",
    selected_commentary[
        "audit"
    ]["issues"]
)

print("\nOUTPUT:\n")

print(
    selected_commentary["text"]
)

COMMENTARY SELECTION
SOURCE: deterministic_fallback
DRAFT AUDIT PASSED: False
ISSUES: ['review_claim_without_citation', 'llm_price_interpretation', 'unsupported_qualitative_claim']

OUTPUT:

Nhận xét tự động từ LLM không vượt qua kiểm tra grounding, vì vậy hệ thống chỉ hiển thị evidence đã truy xuất:
1. SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT — 46 review liên kết; 3 review được truy xuất: [review_0126056], [review_0077221], [review_0008587].
2. Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB) — 33 review liên kết; 3 review được truy xuất: [review_0482507], [review_0374815], [review_0360039].
3. Hugolog 64GB Micro SD Card, Micro SDXC UHS-I Memory Card for LaView Camera– 95MB/s,633X,U3,C10, Full HD Video V30, A1, FAT32, High Speed Flash TF Card P500 for Phone/Tablet/PC/Computer — 2 review liên kết; 2 review được truy xuất: [review_00

In [75]:
# ============================================================
# CELL 66 — FINAL STEP 14 CHATBOT
# ============================================================

def chatbot_step14(
    query,
    user_id=None,
    top_k=5,
    candidate_k=200,
    search_k=1000,
    max_reviews=3
):

    # --------------------------------------------------------
    # 1. KE-aware retrieval + BPR
    # --------------------------------------------------------

    search_result = (
        ke_hybrid_search(
            query=query,
            user_id=user_id,
            top_k=top_k,
            candidate_k=candidate_k,
            search_k=search_k
        )
    )


    # --------------------------------------------------------
    # 2. Effective Knowledge + Review RAG
    # --------------------------------------------------------

    rag_context = (
        build_ke_rag_context(
            search_result,
            max_reviews=max_reviews
        )
    )


    # --------------------------------------------------------
    # 3. Product-review ownership audit
    # --------------------------------------------------------

    relation_audit = (
        audit_ke_review_relations(
            rag_context
        )
    )


    if not relation_audit["passed"]:

        raise RuntimeError(
            "Product-review relation audit failed."
        )


    # --------------------------------------------------------
    # 4. Deterministic product information
    # --------------------------------------------------------

    product_cards = (
        format_ke_product_cards(
            rag_context
        )
    )


    # --------------------------------------------------------
    # 5. LLM draft
    # --------------------------------------------------------

    llm_result = (
        generate_ke_commentary(
            rag_context
        )
    )


    # --------------------------------------------------------
    # 6. Grounding audit + fallback
    # --------------------------------------------------------

    commentary = (
        select_ke_commentary(
            llm_result["text"],
            rag_context
        )
    )


    return {

        "query": query,

        "user_id": user_id,

        "mode": (
            search_result["mode"]
        ),

        "requirements": (
            search_result[
                "requirements"
            ]
        ),

        "active_ke_products": (
            search_result[
                "active_ke_products"
            ]
        ),

        "bpr_known_candidates": (
            search_result[
                "bpr_known_candidates"
            ]
        ),

        "products": (
            rag_context["products"]
        ),

        "product_cards": (
            product_cards
        ),

        "llm_draft": (
            llm_result["text"]
        ),

        "commentary": (
            commentary["text"]
        ),

        "commentary_source": (
            commentary["source"]
        ),

        "commentary_audit": (
            commentary["audit"]
        ),

        "review_relation_audit": (
            relation_audit
        )
    }


print(
    "PASS: STEP 14 CHATBOT FUNCTION READY"
)

PASS: STEP 14 CHATBOT FUNCTION READY


In [76]:
# ============================================================
# CELL 67 — FINAL END-TO-END TEST
# ============================================================

step14_final_result = (
    chatbot_step14(
        query=(
            "microSD 64GB under 20 USD "
            "for photos and video"
        ),

        user_id=TEST_KNOWN_USER,

        top_k=5,

        candidate_k=200,

        search_k=1000,

        max_reviews=3
    )
)


print("=" * 80)
print("STEP 14 — FINAL END-TO-END TEST")
print("=" * 80)

print(
    "QUERY:",
    step14_final_result["query"]
)

print(
    "MODE:",
    step14_final_result["mode"]
)

print(
    "BPR-KNOWN CANDIDATES:",
    step14_final_result[
        "bpr_known_candidates"
    ]
)

print(
    "ACTIVE KE PRODUCTS:",
    step14_final_result[
        "active_ke_products"
    ]
)

print(
    "REVIEW RELATION PASS:",
    step14_final_result[
        "review_relation_audit"
    ]["passed"]
)

print(
    "COMMENTARY SOURCE:",
    step14_final_result[
        "commentary_source"
    ]
)

print(
    "LLM DRAFT AUDIT PASS:",
    step14_final_result[
        "commentary_audit"
    ]["passed"]
)


print("\n" + "=" * 80)
print("PRODUCTS")
print("=" * 80)

print(
    step14_final_result[
        "product_cards"
    ]
)


print("\n" + "=" * 80)
print("LLM DRAFT — INTERNAL ONLY")
print("=" * 80)

print(
    step14_final_result[
        "llm_draft"
    ]
)


print("\n" + "=" * 80)
print("DISPLAYED COMMENTARY")
print("=" * 80)

print(
    step14_final_result[
        "commentary"
    ]
)


# ------------------------------------------------------------
# Critical assertions
# ------------------------------------------------------------

assert (
    step14_final_result["mode"]
    == "hybrid_bpr"
)

assert (
    step14_final_result[
        "review_relation_audit"
    ]["passed"]
)

assert (
    step14_final_result[
        "active_ke_products"
    ] == 0
)

assert (
    len(
        step14_final_result[
            "products"
        ]
    ) == 5
)


print(
    "\nPASS: STEP 14 END-TO-END PIPELINE"
)

STEP 14 — FINAL END-TO-END TEST
QUERY: microSD 64GB under 20 USD for photos and video
MODE: hybrid_bpr
BPR-KNOWN CANDIDATES: 26
ACTIVE KE PRODUCTS: 0
REVIEW RELATION PASS: True
COMMENTARY SOURCE: deterministic_fallback
LLM DRAFT AUDIT PASS: False

PRODUCTS
1. SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
   Product ID: amazon_B0BRZPCQNJ
   Giá dữ liệu: 16.99 USD
   Rating: 4.7/5
   Số rating metadata: 121509
   Review liên kết: 46

2. Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
   Product ID: amazon_B09HKJY2QB
   Giá dữ liệu: 6.49 USD
   Rating: 4.6/5
   Số rating metadata: 15428
   Review liên kết: 33

3. Hugolog 64GB Micro SD Card, Micro SDXC UHS-I Memory Card for LaView Camera– 95MB/s,633X,U3,C10, Full HD Video V30, A1, FAT32, High Speed Flash TF Card P500 for Phone/Tablet/PC/Computer
   Product ID: amazon_B0C1H1Q3C1
   Giá dữ li

In [77]:
# ============================================================
# CELL 68 — SAVE STEP 14 CONFIG
# ============================================================

import os
import json
from datetime import datetime, timezone


STEP14_DIR = "/kaggle/working/step14"

os.makedirs(
    STEP14_DIR,
    exist_ok=True
)


STEP14_CONFIG_PATH = os.path.join(
    STEP14_DIR,
    "step14_config.json"
)


step14_config = {

    "step": "step14_knowledge_editing_integration",

    "status": "completed",

    "created_at": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    # --------------------------------------------------------
    # Knowledge Editing
    # --------------------------------------------------------

    "knowledge_editing": {

        "method": (
            "retrieval_time_knowledge_edit_overlay"
        ),

        "parameter_weight_editing": False,

        "base_dataset_mutated": False,

        "edit_store": (
            "knowledge_edits.json"
        ),

        "editable_fields": [
            "product_name",
            "description",
            "product_features",
            "price",
            "currency",
            "product_url"
        ],

        "protected_fields": [
            "product_id",
            "source",
            "average_rating",
            "review_count",
            "smart_rating",
            "smart_review_count",
            "faiss_row_id",
            "review_id",
            "user_id"
        ],

        "active_controlled_test_edits": 0,

        "controlled_test_edits_preserved_as_history": True
    },

    # --------------------------------------------------------
    # Retrieval
    # --------------------------------------------------------

    "retrieval": {

        "embedding_model": (
            "sentence-transformers/"
            "paraphrase-multilingual-MiniLM-L12-v2"
        ),

        "faiss_index_type": (
            "IndexFlatIP"
        ),

        "similarity": "cosine",

        "search_k": 1000,

        "candidate_k": 200,

        "top_k": 5,

        "ke_vector_overlay": True
    },

    # --------------------------------------------------------
    # Semantic ranking
    # --------------------------------------------------------

    "ranking": {

        "semantic_weight": 0.75,

        "rating_weight": 0.20,

        "review_count_weight": 0.05,

        "constraints": [
            "capacity",
            "max_price",
            "currency"
        ]
    },

    # --------------------------------------------------------
    # BPR
    # --------------------------------------------------------

    "personalization": {

        "model": (
            "BayesianPersonalizedRanking"
        ),

        "requires_known_user": True,

        "requires_known_item": True,

        "fallback": (
            "semantic_ranking"
        ),

        "bpr_adjustment_weight": 0.04,

        "minimum_known_candidates": 2,

        "unknown_items_unchanged": True
    },

    # --------------------------------------------------------
    # Review RAG
    # --------------------------------------------------------

    "review_rag": {

        "enabled": True,

        "candidate_limit_per_product": 300,

        "max_reviews_per_product": 3,

        "retrieval": (
            "semantic_cosine_similarity"
        ),

        "review_ownership_audit": True
    },

    # --------------------------------------------------------
    # LLM
    # --------------------------------------------------------

    "llm": {

        "model": (
            "Qwen/Qwen2.5-3B-Instruct"
        ),

        "quantization": "4bit_nf4",

        "generation": (
            "deterministic"
        ),

        "grounding_rule_audit": True,

        "deterministic_fallback": True
    },

    # --------------------------------------------------------
    # Important limitations
    # --------------------------------------------------------

    "limitations": [

        (
            "Knowledge Editing is retrieval-time "
            "knowledge-base editing, not LLM "
            "parameter-weight editing."
        ),

        (
            "Prices are historical dataset values, "
            "not real-time prices."
        ),

        (
            "No automatic currency conversion."
        ),

        (
            "Eligible candidate counts are measured "
            "within retrieved FAISS candidates plus "
            "the KE overlay, not the full catalog."
        ),

        (
            "Review semantic retrieval considers at "
            "most the first 300 linked reviews per "
            "product."
        ),

        (
            "Review text is truncated before being "
            "used as compact evidence."
        ),

        (
            "Rule-based grounding audit reduces "
            "unsupported output but is not a complete "
            "hallucination detector."
        ),

        (
            "Review ownership audit verifies linkage "
            "only; it does not prove claim faithfulness."
        ),

        (
            "BPR personalization only affects known "
            "users and BPR-known candidate products."
        )
    ]
}


with open(
    STEP14_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step14_config,
        f,
        ensure_ascii=False,
        indent=2
    )


print("=" * 80)
print("STEP 14 CONFIG SAVED")
print("=" * 80)

print(
    "PATH:",
    STEP14_CONFIG_PATH
)

print(
    "METHOD:",
    step14_config[
        "knowledge_editing"
    ]["method"]
)

print(
    "PARAMETER EDITING:",
    step14_config[
        "knowledge_editing"
    ][
        "parameter_weight_editing"
    ]
)

print(
    "\nPASS: STEP14 CONFIG SAVED"
)

STEP 14 CONFIG SAVED
PATH: /kaggle/working/step14/step14_config.json
METHOD: retrieval_time_knowledge_edit_overlay
PARAMETER EDITING: False

PASS: STEP14 CONFIG SAVED


In [78]:
# ============================================================
# CELL 69 — SAVE FINAL TEST
# ============================================================

STEP14_FINAL_PATH = os.path.join(
    STEP14_DIR,
    "step14_final_test.json"
)


final_products_summary = []


for rank, product in enumerate(
    step14_final_result["products"],
    start=1
):

    final_products_summary.append({

        "rank": rank,

        "product_id": (
            product["product_id"]
        ),

        "title": (
            product["title"]
        ),

        "price": (
            product["price"]
        ),

        "currency": (
            product["currency"]
        ),

        "rating": (
            product[
                "product_average_rating"
            ]
        ),

        "metadata_rating_count": (
            product[
                "metadata_rating_count"
            ]
        ),

        "linked_review_count": (
            product[
                "linked_review_count"
            ]
        ),

        "retrieved_review_ids": [
            r["review_id"]
            for r in product["reviews"]
        ],

        "knowledge_edit_applied": (
            product[
                "knowledge_edit"
            ]["applied"]
        ),

        "bpr_known": (
            product[
                "ranking"
            ]["bpr_known"]
        ),

        "bpr_adjustment": (
            product[
                "ranking"
            ]["bpr_adjustment"]
        ),

        "hybrid_score": (
            product[
                "ranking"
            ]["hybrid_score"]
        )
    })


step14_final_test = {

    "step": (
        "step14_knowledge_editing_integration"
    ),

    "status": (
        "end_to_end_test_completed"
    ),

    "query": (
        step14_final_result["query"]
    ),

    "user_id": (
        step14_final_result["user_id"]
    ),

    "mode": (
        step14_final_result["mode"]
    ),

    "requirements": (
        step14_final_result[
            "requirements"
        ]
    ),

    "bpr_known_candidates": (
        step14_final_result[
            "bpr_known_candidates"
        ]
    ),

    "active_ke_products": (
        step14_final_result[
            "active_ke_products"
        ]
    ),

    "review_relation_audit": (
        step14_final_result[
            "review_relation_audit"
        ]
    ),

    # Important:
    # this is the audit of the LLM draft
    "llm_draft_audit": (
        step14_final_result[
            "commentary_audit"
        ]
    ),

    "llm_draft": (
        step14_final_result[
            "llm_draft"
        ]
    ),

    "displayed_commentary_source": (
        step14_final_result[
            "commentary_source"
        ]
    ),

    "displayed_commentary": (
        step14_final_result[
            "commentary"
        ]
    ),

    "products": (
        final_products_summary
    ),

    "interpretation": {

        "llm_draft_accepted": (
            step14_final_result[
                "commentary_source"
            ] == "llm"
        ),

        "fallback_triggered": (
            step14_final_result[
                "commentary_source"
            ]
            == "deterministic_fallback"
        ),

        "controlled_test_edits_active": False,

        "knowledge_edit_method": (
            "retrieval_time_knowledge_edit_overlay"
        ),

        "parameter_weight_editing": False
    }
}


with open(
    STEP14_FINAL_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step14_final_test,
        f,
        ensure_ascii=False,
        indent=2,
        default=str
    )


print("=" * 80)
print("STEP 14 FINAL TEST SAVED")
print("=" * 80)

print(
    "PATH:",
    STEP14_FINAL_PATH
)

print(
    "MODE:",
    step14_final_test["mode"]
)

print(
    "PRODUCTS:",
    len(
        step14_final_test[
            "products"
        ]
    )
)

print(
    "LLM ACCEPTED:",
    step14_final_test[
        "interpretation"
    ][
        "llm_draft_accepted"
    ]
)

print(
    "FALLBACK:",
    step14_final_test[
        "interpretation"
    ][
        "fallback_triggered"
    ]
)

print(
    "\nPASS: FINAL TEST SAVED"
)

STEP 14 FINAL TEST SAVED
PATH: /kaggle/working/step14/step14_final_test.json
MODE: hybrid_bpr
PRODUCTS: 5
LLM ACCEPTED: False
FALLBACK: True

PASS: FINAL TEST SAVED


In [79]:
# ============================================================
# CELL 70 — STEP 14 ARTIFACT VALIDATION
# ============================================================

import os
import json


required_artifacts = [
    "knowledge_edits.json",
    "step14_ke_evaluation.json",
    "step14_config.json",
    "step14_final_test.json"
]


print("=" * 80)
print("STEP 14 ARTIFACT VALIDATION")
print("=" * 80)


all_exist = True


for filename in required_artifacts:

    path = os.path.join(
        STEP14_DIR,
        filename
    )

    exists = os.path.exists(
        path
    )

    size = (
        os.path.getsize(path)
        if exists
        else 0
    )

    print(
        f"{filename:<32}",
        "OK" if exists else "MISSING",
        f"{size:,} bytes"
    )

    if not exists:
        all_exist = False


assert all_exist


# ------------------------------------------------------------
# Reload from disk
# ------------------------------------------------------------

with open(
    STEP14_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    check_config = json.load(f)


with open(
    STEP14_FINAL_PATH,
    "r",
    encoding="utf-8"
) as f:

    check_final = json.load(f)


KE_EVAL_PATH = os.path.join(
    STEP14_DIR,
    "step14_ke_evaluation.json"
)


with open(
    KE_EVAL_PATH,
    "r",
    encoding="utf-8"
) as f:

    check_ke_eval = json.load(f)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert (
    check_config["status"]
    == "completed"
)

assert (
    check_config[
        "knowledge_editing"
    ][
        "parameter_weight_editing"
    ]
    is False
)

assert (
    check_final["status"]
    == "end_to_end_test_completed"
)

assert (
    check_final["mode"]
    == "hybrid_bpr"
)

assert (
    check_final[
        "active_ke_products"
    ] == 0
)

assert (
    check_final[
        "review_relation_audit"
    ]["passed"]
)

assert (
    len(
        check_final["products"]
    ) == 5
)

assert (
    check_final[
        "displayed_commentary_source"
    ]
    in [
        "llm",
        "deterministic_fallback"
    ]
)


print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print(
    "KE METHOD:",
    check_config[
        "knowledge_editing"
    ]["method"]
)

print(
    "PARAMETER EDITING:",
    check_config[
        "knowledge_editing"
    ][
        "parameter_weight_editing"
    ]
)

print(
    "CONTROLLED SEMANTIC TEST:",
    check_ke_eval.get(
        "similarity_improved"
    )
)

print(
    "FINAL MODE:",
    check_final["mode"]
)

print(
    "BPR-KNOWN CANDIDATES:",
    check_final[
        "bpr_known_candidates"
    ]
)

print(
    "ACTIVE KE PRODUCTS:",
    check_final[
        "active_ke_products"
    ]
)

print(
    "REVIEW RELATION PASS:",
    check_final[
        "review_relation_audit"
    ]["passed"]
)

print(
    "LLM DRAFT AUDIT:",
    check_final[
        "llm_draft_audit"
    ]["passed"]
)

print(
    "DISPLAYED SOURCE:",
    check_final[
        "displayed_commentary_source"
    ]
)

print(
    "\nPASS: STEP 14 COMPLETED"
)

STEP 14 ARTIFACT VALIDATION
knowledge_edits.json             OK 2,142 bytes
step14_ke_evaluation.json        OK 1,285 bytes
step14_config.json               OK 2,732 bytes
step14_final_test.json           OK 6,011 bytes

FINAL SUMMARY
KE METHOD: retrieval_time_knowledge_edit_overlay
PARAMETER EDITING: False
CONTROLLED SEMANTIC TEST: None
FINAL MODE: hybrid_bpr
BPR-KNOWN CANDIDATES: 26
ACTIVE KE PRODUCTS: 0
REVIEW RELATION PASS: True
LLM DRAFT AUDIT: False
DISPLAYED SOURCE: deterministic_fallback

PASS: STEP 14 COMPLETED
